In [1]:
# ==========================================
# 02_multi_omics_ef_lf_cv_rebuild
# EF/LF comparison + top10 late-fusion search
# ==========================================

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.linear_model import Lasso, ElasticNet, Ridge
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import GroupKFold, GridSearchCV, ParameterGrid, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression

from scipy import sparse
from joblib import Parallel, delayed
import traceback


In [2]:
# ==========================================
# EF paper-style v13 tp/model refined tuning
# HELD-OUT TEST notebook
# ==========================================

DATA_ROOT = Path("../../DifferentCom_data_rebuild")
TP_DIR = DATA_ROOT / "tp_views"
BASE_DIR = DATA_ROOT / "base_patient_tables"

# Must match the EF v13 TRAINING notebook exactly.
TRAIN_EXPERIMENT_TAG = "ef_paper_style_v13_tp_model_refined_tuning"
TRAIN_RESULT_DIR = DATA_ROOT / f"results_multi_omics_ef_train_only_{TRAIN_EXPERIMENT_TAG}"
TRAIN_SUMMARY_PATH = TRAIN_RESULT_DIR / f"multi_omics_ef_summary_{TRAIN_EXPERIMENT_TAG}.csv"

# Optional: if the training notebook saved this table, this is the paper-style reference table.
TRAIN_BEST_BY_TISSUE_TP_PATH = (
    TRAIN_RESULT_DIR
    / "grouped_summary"
    / f"paper_style_best_by_tissue_tp_{TRAIN_EXPERIMENT_TAG}.csv"
)

# Separate held-out test output tag.
TEST_EXPERIMENT_TAG = "ef_final_test_paper_style_v13_tp_model_refined_tuning"
SAVE_DIR = DATA_ROOT / "testResult" / f"results_multi_omics_ef_test_{TEST_EXPERIMENT_TAG}"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

TARGET_PATH = BASE_DIR / "target_by_patient.csv"

TISSUES = ["csf", "ser"]
TIMEPOINTS = [24, 48, 72, 96, 120]
COMBOS = ["ABC"]

# Must match EF v13 training model family.
MODEL_TYPES = [
    "ridge",
    "elasticnet",
    "pls",
    "svr_linear",
    "gbr",
]

CV_SEARCH_MODE = "nested_inner3_v13_training_matched_heldout_test"


def get_ef_tuning_group(tissue: str, tp: int) -> str:
    """
    Same tuning-group rule as EF v13 training.
    """
    tissue = str(tissue).lower()
    tp = int(tp)

    if (tissue == "csf" and tp in [72, 96]) or (tissue == "ser" and tp == 96):
        return "strong"
    if (tissue == "csf" and tp in [24, 48, 120]) or (tissue == "ser" and tp == 120):
        return "middle"
    return "weak"


def get_ef_model_types_for_job(tissue: str, tp: int) -> list[str]:
    """
    Same v13 result-based model routing as training.
    Mainly used for traceability/checking; held-out test candidates are selected
    from the v13 training-CV summary, not from test performance.
    """
    tissue = str(tissue).lower()
    tp = int(tp)

    if tissue == "csf":
        if tp == 24:
            candidates = ["ridge", "elasticnet"]
        elif tp == 48:
            candidates = ["ridge", "svr_linear"]
        elif tp == 72:
            candidates = ["ridge", "elasticnet", "svr_linear"]
        elif tp == 96:
            candidates = ["ridge", "elasticnet", "pls", "svr_linear", "gbr"]
        elif tp == 120:
            candidates = ["ridge"]
        else:
            candidates = ["ridge"]

    elif tissue == "ser":
        if tp in [24, 48, 72]:
            candidates = ["ridge"]
        elif tp == 96:
            candidates = ["ridge", "elasticnet", "pls", "gbr"]
        elif tp == 120:
            candidates = ["ridge", "elasticnet"]
        else:
            candidates = ["ridge"]
    else:
        candidates = ["ridge"]

    return [m for m in candidates if m in MODEL_TYPES]


def get_ef_clinical_modes_for_job(tissue: str, tp: int) -> list[str]:
    """
    Same v13 clinical routing as training: evaluate both omics_only and light.
    """
    return CLINICAL_MODES_TO_RUN.copy()


# Paper-style comparison:
# - omics_only: omics features only
# - light: omics + Age/Gender/Level
CLINICAL_MODES_TO_RUN = ["omics_only", "light"]

# Must match EF v13 training feature-selection mode.
FEATURE_SELECTION_MODES_TO_RUN = [
    "f_regression_topk",
]

USE_BLOCKWISE_EF = True
BLOCKWISE_MIN_PER_BLOCK = 5
EF_MAX_FINAL_SELECTED = 70
EF_MAX_FINAL_SELECTED_LATE = 40

MIN_FEATURE_FREQ = 3
MIN_FEATURE_FREQ_LATE = 3
LATE_TP_START = 96

USE_DELTA_VIEW = False
LIGHT_CLIN_COLS = ["Age", "Gender", "Level"]

N_SPLITS_OUTER = 5
N_SPLITS_INNER = 3

USE_STRATIFIED_OUTER_CV = True
USE_STRATIFIED_INNER_CV = True
N_Y_BINS_FOR_OUTER_CV = 3
OUTER_CV_RANDOM_STATE = 42

LOW_VALID_SST_WARN_THRESHOLD = 200.0

USE_FEATURE_WEIGHTING = True
FEATURE_WEIGHT_MODES = ["none", "soft"] if USE_FEATURE_WEIGHTING else ["none"]

FEATURE_WEIGHT_MIN = 1.0
FEATURE_WEIGHT_MAX = 1.20
FEATURE_WEIGHT_DEFAULT = 1.0
FEATURE_WEIGHT_TOPK_HIGH = 1.20
FEATURE_WEIGHT_TOPK_LOW = 1.0

TOP_N = 10
COMPUTE_TRAIN_INNER_OOF = False

MIN_OBS_FRAC = 0.40
SPARSE_FALLBACK_THRESHOLDS = [0.40, 0.35]

MIN_FEATURES_AFTER_FILTER = 15
MIN_OMICS_FEATURES_AFTER_FILTER = 60

# Which training-CV configs should be projected into the held-out test set?
# Recommended: one best config per tissue/timepoint using v13 training OOF only.
TEST_CONFIG_SELECTION_MODE = "best_per_tissue_tp"

# Used only when TEST_CONFIG_SELECTION_MODE == "top_n_overall"
N_TEST_CONFIGS_OVERALL = 10

# Selection metric for final held-out test candidates.
# Important: this is training-CV OOF only; test_r2 is never used for candidate selection.
TEST_SELECTION_PRIMARY_METRIC = "oof_r2"

print("TRAIN_SUMMARY_PATH:", TRAIN_SUMMARY_PATH.resolve())
print("TRAIN_BEST_BY_TISSUE_TP_PATH:", TRAIN_BEST_BY_TISSUE_TP_PATH.resolve())
print("SAVE_DIR:", SAVE_DIR.resolve())
print("TRAIN_EXPERIMENT_TAG:", TRAIN_EXPERIMENT_TAG)
print("TEST_EXPERIMENT_TAG:", TEST_EXPERIMENT_TAG)
print("CV_SEARCH_MODE:", CV_SEARCH_MODE)
print("MODEL_TYPES:", MODEL_TYPES)
print("EF_TUNING_GROUPS:", {(t,tp): get_ef_tuning_group(t,tp) for t in TISSUES for tp in TIMEPOINTS})
print("EF_MODEL_ROUTING:", {(t,tp): get_ef_model_types_for_job(t,tp) for t in TISSUES for tp in TIMEPOINTS})
print("EF_CLINICAL_ROUTING:", {(t,tp): get_ef_clinical_modes_for_job(t,tp) for t in TISSUES for tp in TIMEPOINTS})
print("CLINICAL_MODES_TO_RUN:", CLINICAL_MODES_TO_RUN)
print("FEATURE_SELECTION_MODES_TO_RUN:", FEATURE_SELECTION_MODES_TO_RUN)
print("USE_FEATURE_WEIGHTING:", USE_FEATURE_WEIGHTING)
print("FEATURE_WEIGHT_MODES:", FEATURE_WEIGHT_MODES)
print("TEST_SELECTION_PRIMARY_METRIC:", TEST_SELECTION_PRIMARY_METRIC)



TRAIN_SUMMARY_PATH: /data2/jinoh001/stable/research_SPI/DifferentCom_data_rebuild/results_multi_omics_ef_train_only_ef_paper_style_v13_tp_model_refined_tuning/multi_omics_ef_summary_ef_paper_style_v13_tp_model_refined_tuning.csv
TRAIN_BEST_BY_TISSUE_TP_PATH: /data2/jinoh001/stable/research_SPI/DifferentCom_data_rebuild/results_multi_omics_ef_train_only_ef_paper_style_v13_tp_model_refined_tuning/grouped_summary/paper_style_best_by_tissue_tp_ef_paper_style_v13_tp_model_refined_tuning.csv
SAVE_DIR: /data2/jinoh001/stable/research_SPI/DifferentCom_data_rebuild/testResult/results_multi_omics_ef_test_ef_final_test_paper_style_v13_tp_model_refined_tuning
TRAIN_EXPERIMENT_TAG: ef_paper_style_v13_tp_model_refined_tuning
TEST_EXPERIMENT_TAG: ef_final_test_paper_style_v13_tp_model_refined_tuning
CV_SEARCH_MODE: nested_inner3_v13_training_matched_heldout_test
MODEL_TYPES: ['ridge', 'elasticnet', 'pls', 'svr_linear', 'gbr']
EF_TUNING_GROUPS: {('csf', 24): 'middle', ('csf', 48): 'middle', ('csf', 72

In [3]:
def read_id_txt(path: str) -> list[str]:
    df = pd.read_csv(path)
    return df["Patient"].astype(str).tolist()




In [4]:
TRAIN_ID_PATH = "../../data/training_id.txt"
TEST_ID_PATH = "../../data/testing_id.txt"

TRAIN_IDS = read_id_txt(TRAIN_ID_PATH)
TEST_IDS = read_id_txt(TEST_ID_PATH)

train_set = set(map(str, TRAIN_IDS))
test_set = set(map(str, TEST_IDS))
overlap = sorted(train_set.intersection(test_set))

if len(overlap) > 0:
    raise ValueError(
        f"TRAIN_IDS and TEST_IDS overlap: n={len(overlap)}, examples={overlap[:10]}"
    )

print("n_train_ids:", len(TRAIN_IDS))
print("n_test_ids:", len(TEST_IDS))
print("n_overlap:", len(overlap))


n_train_ids: 60
n_test_ids: 30
n_overlap: 0


In [5]:
# ==========================================
# Loader / helper
# ==========================================
def load_feature_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(
        path,
        index_col=0,
        na_values=["", " ", "NA", "N/A", "nan", "NaN", ".", "-"],
        keep_default_na=True,
    )
    df.index = df.index.astype(str)

    # whitespace-only string도 NaN 처리
    df = df.replace(r"^\s*$", np.nan, regex=True)

    sentinel_values = [
        -10.210340372,
        -1.127439639,
        -1.128465252,
    ]
    df = df.replace(sentinel_values, np.nan)

    # clinical columns는 merge_with_light_clinical에서 붙으므로,
    # 여기 들어온 feature CSV는 기본적으로 omics라고 보고 numeric coercion.
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


def load_target(path: Path) -> pd.Series:
    y = pd.read_csv(path, index_col=0).iloc[:, 0]
    y.index = y.index.astype(str)
    y.name = "DeltaTMS"
    return y


def build_feature_path(tissue: str, combo: str, tp: int, use_delta: bool = False) -> Path:
    if use_delta:
        return TP_DIR / f"x_omicsdelta_{tissue}_{combo}_{tp}.csv"
    return TP_DIR / f"x_omics_{tissue}_{combo}_{tp}.csv"


def build_clinical_path(tissue: str, combo: str, tp: int) -> Path:
    return TP_DIR / f"x_clinical_{tissue}_{combo}_{tp}.csv"


def load_light_clinical_csv(path: Path) -> pd.DataFrame:
    clin = pd.read_csv(path, index_col=0)
    clin.index = clin.index.astype(str)

    missing_cols = [c for c in LIGHT_CLIN_COLS if c not in clin.columns]
    if len(missing_cols) > 0:
        raise ValueError(f"clinical file missing required cols: {missing_cols}")

    return clin[LIGHT_CLIN_COLS].copy()


def merge_with_light_clinical(
    X_feat: pd.DataFrame,
    tissue: str,
    combo: str,
    tp: int,
) -> pd.DataFrame:
    clin_path = build_clinical_path(tissue=tissue, combo=combo, tp=tp)
    if not clin_path.exists():
        raise FileNotFoundError(f"clinical file not found: {clin_path}")

    X_clin = load_light_clinical_csv(clin_path)

    common_idx = X_feat.index.intersection(X_clin.index)
    X_feat2 = X_feat.loc[common_idx].copy()
    X_clin2 = X_clin.loc[common_idx].copy()

    overlap_cols = [c for c in X_clin2.columns if c in X_feat2.columns]
    if len(overlap_cols) > 0:
        X_feat2 = X_feat2.drop(columns=overlap_cols)

    X_out = pd.concat([X_feat2, X_clin2], axis=1)
    return X_out


def align_xy(X: pd.DataFrame, y: pd.Series) -> tuple[pd.DataFrame, pd.Series]:
    idx = X.index.intersection(y.index)
    return X.loc[idx].copy(), y.loc[idx].copy()


In [6]:
# ==========================================
# Fold-safe sparse filter / preprocess / model
# ==========================================

def make_preprocess(X: pd.DataFrame) -> ColumnTransformer:
    cat_cols = [c for c in X.columns if c in ["Gender", "Level", "AIS"]]
    num_cols = [c for c in X.columns if c not in cat_cols]

    return ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]), num_cols),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("ohe", OneHotEncoder(handle_unknown="ignore")),
            ]), cat_cols),
        ],
        remainder="drop"
    )



def make_model(model_type: str):
    if model_type == "lasso":
        return Lasso(max_iter=50000, tol=1e-3, random_state=42)
    elif model_type == "elasticnet":
        return ElasticNet(max_iter=50000, tol=1e-3, random_state=42)
    elif model_type == "ridge":
        return Ridge()
    elif model_type == "pls":
        return PLSRegression(scale=False)
    elif model_type == "svr_linear":
        return SVR(kernel="linear")
    elif model_type == "svr_rbf":
        return SVR(kernel="rbf")
    elif model_type == "extratrees":
        return ExtraTreesRegressor(
            random_state=42,
            n_jobs=1,
        )
    elif model_type == "gbr":
        return GradientBoostingRegressor(
            random_state=42,
        )
    else:
        raise ValueError(model_type)


def get_model_params(params: dict) -> dict:
    """Remove search-only parameters before passing params into the sklearn model."""
    search_only = {
        "prefilter_k",
        "weight_mode",
        "top_k",
        "feature_selection_mode",
    }
    return {k: v for k, v in params.items() if k not in search_only}


def make_param_grid(model_type: str) -> dict:
    # This legacy pipeline grid is kept for compatibility with older helper functions.
    # The paper-style weighted search uses the explicit param_space in fit_best_weighted_model().
    if model_type == "lasso":
        return {
            "model__alpha": [0.05, 0.1, 0.2, 0.5, 1.0],
            "select__k": [40, 60, 80],
        }
    elif model_type == "elasticnet":
        return {
            "model__alpha": [0.05, 0.1, 0.2, 0.5, 1.0],
            "model__l1_ratio": [0.1, 0.2, 0.4, 0.6],
            "select__k": [40, 60, 80],
        }
    elif model_type == "ridge":
        return {
            "model__alpha": [5.0, 10.0, 20.0, 50.0, 100.0],
            "select__k": [60, 80, 100],
        }
    elif model_type == "pls":
        return {
            "model__n_components": [2, 3, 5],
            "select__k": [40, 60, 80],
        }
    elif model_type in ["svr_linear", "svr_rbf"]:
        return {
            "model__C": [0.1, 1.0, 10.0],
            "model__epsilon": [0.1, 1.0],
            "select__k": [40, 60, 80],
        }
    elif model_type == "extratrees":
        return {
            "model__n_estimators": [100, 200],
            "model__max_depth": [2, 3, None],
            "model__min_samples_leaf": [2, 4],
            "select__k": [40, 60, 80],
        }
    elif model_type == "gbr":
        return {
            "model__n_estimators": [50, 100],
            "model__learning_rate": [0.03, 0.05],
            "model__max_depth": [1, 2],
            "model__min_samples_leaf": [2, 4],
            "select__k": [40, 60, 80],
        }
    else:
        raise ValueError(model_type)


def make_pipeline(X: pd.DataFrame, model_type: str) -> Pipeline:
    return Pipeline([
        ("preprocess", make_preprocess(X)),
        ("select", SelectKBest(score_func=f_regression)),
        ("model", make_model(model_type)),
    ])


In [7]:

# ==========================================
# Small-n safety helper
# ==========================================
def sanitize_model_params(model_type: str, model_params: dict, X_fit: pd.DataFrame) -> dict:
    """
    PLSRegression requires n_components <= min(n_samples - 1, n_features).
    Some folds can have very few selected features after fold-safe filtering,
    so cap n_components at fit time instead of failing the whole experiment.
    """
    params = dict(model_params)

    if model_type == "pls" and "n_components" in params:
        max_comp = int(max(1, min(X_fit.shape[0] - 1, X_fit.shape[1])))
        params["n_components"] = int(min(int(params["n_components"]), max_comp))

    return params


In [8]:
def split_omics_blocks_transformed(columns, protected_keep_cols=None):
    """
    preprocess 이후 컬럼명 기준으로 PROT / MET / RNA / CLIN 구분.
    실제 transformed name 예:
      num__CSF_PROT_...
      num__SER_MET_...
      num__RNA_...
      num__Age
      cat__Gender_M
      cat__Level_C
    """
    if protected_keep_cols is None:
        protected_keep_cols = []

    protected_keep_cols = set(protected_keep_cols)

    prot_cols = []
    met_cols = []
    rna_cols = []
    clin_cols = []

    for c in columns:
        c_str = str(c)
        c_low = c_str.lower()

        # transformed clinical
        if any(
            
            c_low == f"num__{x.lower()}" or c_low.startswith(f"cat__{x.lower()}_")
            for x in protected_keep_cols
        ):
            clin_cols.append(c)
            continue

        # transformed omics
        if "prot_" in c_low:
            prot_cols.append(c)
        elif "met_" in c_low:
            met_cols.append(c)
        elif "rna_" in c_low:
            rna_cols.append(c)

    return {
        "PROT": prot_cols,
        "MET": met_cols,
        "RNA": rna_cols,
        "CLIN": clin_cols,
    }


In [9]:
def filter_sparse_features_train_test(
    X_train: pd.DataFrame,
    X_valid: pd.DataFrame,
    min_obs_frac: float = 0.70,
    protected_keep_cols: list[str] | None = None,
    fallback_thresholds: list[float] | None = None,
    min_omics_features: int = 50,
):
    """
    Fold-safe sparse filter.
    1) train 기준으로 all-missing / constant column 제거
    2) train 기준 관측률 threshold 적용
    3) protected clinical columns는 강제 유지
    """
    if protected_keep_cols is None:
        protected_keep_cols = []

    if fallback_thresholds is None:
        fallback_thresholds = [min_obs_frac, 0.60, 0.50, 0.40, 0.30, 0.20]

    thresholds = sorted(set(float(x) for x in fallback_thresholds), reverse=True)

    base_cols = list(X_train.columns)
    omics_cols = [c for c in base_cols if c not in protected_keep_cols]

    # 1) train 기준 all-missing 제거
    non_all_missing_cols = X_train.columns[X_train.notna().any(axis=0)].tolist()

    # 2) train 기준 constant 제거
    constant_cols = []
    for c in non_all_missing_cols:
        s = X_train[c]
        nunq = s.dropna().nunique()
        if nunq <= 1:
            constant_cols.append(c)

    candidate_cols = [c for c in non_all_missing_cols if c not in constant_cols]

    # protected clinical은 있으면 항상 유지
    for c in protected_keep_cols:
        if c in X_train.columns and c not in candidate_cols:
            candidate_cols.append(c)

    candidate_cols = [c for c in X_train.columns if c in candidate_cols]

    chosen_thresh = None
    chosen_keep_cols = None

    X_train_base = X_train[candidate_cols].copy()
    X_valid_base = X_valid[candidate_cols].copy()

    for thresh in thresholds:
        obs_rate = X_train_base.notna().mean(axis=0)
        keep_cols = obs_rate[obs_rate >= thresh].index.tolist()

        for c in protected_keep_cols:
            if c in X_train_base.columns and c not in keep_cols:
                keep_cols.append(c)

        keep_cols = [c for c in X_train_base.columns if c in keep_cols]
        omics_keep_cols = [c for c in keep_cols if c not in protected_keep_cols]

        block_map_tmp = split_omics_blocks_transformed(
            X_train_base[keep_cols].columns,
            protected_keep_cols=protected_keep_cols,
        )

        enough_total = len(omics_keep_cols) >= min_omics_features
        enough_blocks = (
            len(block_map_tmp["PROT"]) >= 20 and
            len(block_map_tmp["MET"]) >= 20
        )

        if enough_total and enough_blocks:
            chosen_thresh = thresh
            chosen_keep_cols = keep_cols
            break
    
    if chosen_keep_cols is None:
        chosen_thresh = thresholds[-1]
        obs_rate = X_train_base.notna().mean(axis=0)
        chosen_keep_cols = obs_rate[obs_rate >= chosen_thresh].index.tolist()

        for c in protected_keep_cols:
            if c in X_train_base.columns and c not in chosen_keep_cols:
                chosen_keep_cols.append(c)

        chosen_keep_cols = [c for c in X_train_base.columns if c in chosen_keep_cols]

    X_train_f = X_train_base.loc[:, chosen_keep_cols].copy()
    X_valid_f = X_valid_base.loc[:, chosen_keep_cols].copy()

    omics_keep_cols = [c for c in chosen_keep_cols if c not in protected_keep_cols]
    protected_kept = [c for c in chosen_keep_cols if c in protected_keep_cols]

    info = {
        "chosen_thresh": float(chosen_thresh),
        "n_total_before": int(X_train.shape[1]),
        "n_total_after": int(X_train_f.shape[1]),
        "n_omics_after": int(len(omics_keep_cols)),
        "n_protected_after": int(len(protected_kept)),
        "n_all_missing_removed": int(len([c for c in base_cols if c not in non_all_missing_cols])),
        "n_constant_removed": int(len(constant_cols)),
        "kept_columns": chosen_keep_cols,
    }

    return X_train_f, X_valid_f, info

In [10]:
def drop_train_rows_with_no_omics(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    protected_keep_cols: list[str] | None = None,
):
    """
    train fold에서 omics 값이 하나도 없는 row 제거.
    valid/test는 제거하지 않음.
    """
    if protected_keep_cols is None:
        protected_keep_cols = []

    omics_cols = [c for c in X_train.columns if c not in protected_keep_cols]

    if len(omics_cols) == 0:
        info = {
            "n_train_before": int(len(X_train)),
            "n_train_after": int(len(X_train)),
            "n_removed": 0,
        }
        return X_train.copy(), y_train.copy(), info

    row_has_any_omics = X_train[omics_cols].notna().any(axis=1)
    X_out = X_train.loc[row_has_any_omics].copy()
    y_out = y_train.loc[X_out.index].copy()

    info = {
        "n_train_before": int(len(X_train)),
        "n_train_after": int(len(X_out)),
        "n_removed": int((~row_has_any_omics).sum()),
    }
    return X_out, y_out, info

In [11]:
def prepare_train_valid_fold(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_valid: pd.DataFrame,
    protected_keep_cols: list[str] | None = None,
    min_obs_frac: float = 0.70,
    fallback_thresholds: list[float] | None = None,
    min_omics_features: int = 50,
):
    """
    inner/outer 공통 fold-safe preparation
    1) train에서 omics all-missing row 제거
    2) train 기준 sparse filtering
    3) valid는 row 제거 없이 train 기준 컬럼만 맞춤
    """
    if protected_keep_cols is None:
        protected_keep_cols = []

    X_train2, y_train2, row_info = drop_train_rows_with_no_omics(
        X_train=X_train,
        y_train=y_train,
        protected_keep_cols=protected_keep_cols,
    )

    X_train_f, X_valid_f, sparse_info = filter_sparse_features_train_test(
        X_train=X_train2,
        X_valid=X_valid,
        min_obs_frac=min_obs_frac,
        protected_keep_cols=protected_keep_cols,
        fallback_thresholds=fallback_thresholds,
        min_omics_features=min_omics_features,
    )

    if X_train_f.shape[1] < MIN_FEATURES_AFTER_FILTER:
        raise ValueError(
            f"too few total features after sparse filtering: {X_train_f.shape[1]}"
        )

    if sparse_info["n_omics_after"] < min_omics_features:
        raise ValueError(
            f"too few omics features after sparse filtering: {sparse_info['n_omics_after']}"
        )

    info = {
        "row_info": row_info,
        "sparse_info": sparse_info,
    }
    return X_train_f, y_train2, X_valid_f, info

In [12]:
def build_local_feature_score_df(
    model,
    selected_features: list[str],
) -> pd.DataFrame:
    """
    한 split 안에서 선택된 feature만 가지고 local feature score 생성.
    leakage 없이 split-local weighting에 사용.
    """
    if len(selected_features) == 0:
        return pd.DataFrame(columns=["feature", "selection_freq", "mean_abs_coef", "feature_score"])

    if hasattr(model, "coef_"):
        coef_abs = np.abs(np.ravel(model.coef_))
        if len(coef_abs) != len(selected_features):
            coef_abs = np.ones(len(selected_features), dtype=float)
    elif hasattr(model, "feature_importances_"):
        coef_abs = np.abs(np.ravel(model.feature_importances_))
        if len(coef_abs) != len(selected_features):
            coef_abs = np.ones(len(selected_features), dtype=float)
    else:
        # Nonlinear models such as RBF-SVR do not expose a direct coefficient.
        # In that case the split-local feature score falls back to equal score.
        coef_abs = np.ones(len(selected_features), dtype=float)

    df = pd.DataFrame({
        "feature": selected_features,
        "selection_freq": 1.0,
        "mean_abs_coef": coef_abs,
    })

    max_coef = max(float(df["mean_abs_coef"].max()), 1e-12)
    df["feature_score"] = df["mean_abs_coef"] / max_coef
    df = df.sort_values("feature_score", ascending=False).reset_index(drop=True)
    return df

def split_omics_blocks(columns, protected_keep_cols=None):
    if protected_keep_cols is None:
        protected_keep_cols = []

    protected_keep_cols = set(protected_keep_cols)

    prot_cols = [c for c in columns if c.startswith("PROT_") and c not in protected_keep_cols]
    met_cols  = [c for c in columns if c.startswith("MET_") and c not in protected_keep_cols]
    rna_cols  = [c for c in columns if c.startswith("RNA_") and c not in protected_keep_cols]
    clin_cols = [c for c in columns if c in protected_keep_cols]

    return {
        "PROT": prot_cols,
        "MET": met_cols,
        "RNA": rna_cols,
        "CLIN": clin_cols,
    }


def allocate_blockwise_k(block_map, total_k, min_per_block=10):
    """
    total_k를 PROT/MET/RNA block에 비례 배분.
    block size가 작은 경우 가능한 범위 내에서만 배정.
    """
    omics_names = ["PROT", "MET", "RNA"]
    sizes = {k: len(block_map.get(k, [])) for k in omics_names}
    active = [k for k in omics_names if sizes[k] > 0]

    if total_k is None or len(active) == 0:
        return {k: sizes[k] for k in omics_names}

    total_k = int(total_k)
    total_available = sum(sizes[k] for k in active)
    total_k = min(total_k, total_available)

    alloc = {k: 0 for k in omics_names}

    # 1) active block에 최소 quota 부여
    for k in active:
        alloc[k] = min(min_per_block, sizes[k])

    used = sum(alloc.values())

    # total_k가 너무 작으면 최소 quota를 다시 줄임
    if used > total_k:
        alloc = {k: 0 for k in omics_names}
        base = max(1, total_k // len(active))
        rem = total_k

        for k in active:
            take = min(base, sizes[k], rem)
            alloc[k] = take
            rem -= take

        for k in sorted(active, key=lambda x: sizes[x], reverse=True):
            if rem <= 0:
                break
            extra = min(sizes[k] - alloc[k], rem)
            alloc[k] += extra
            rem -= extra

        return alloc

    # 2) 남은 quota를 block size 비례로 배분
    remaining = total_k - used
    if remaining > 0:
        size_sum = sum(sizes[k] for k in active)
        for k in active:
            extra = int(round(remaining * sizes[k] / size_sum))
            alloc[k] += extra

        # overflow / underflow 정리
        for k in active:
            alloc[k] = min(alloc[k], sizes[k])

        cur = sum(alloc.values())

        if cur < total_k:
            for k in sorted(active, key=lambda x: sizes[x] - alloc[x], reverse=True):
                if cur >= total_k:
                    break
                room = sizes[k] - alloc[k]
                take = min(room, total_k - cur)
                alloc[k] += take
                cur += take

        elif cur > total_k:
            for k in sorted(active, key=lambda x: alloc[x], reverse=True):
                if cur <= total_k:
                    break
                drop = min(alloc[k], cur - total_k)
                alloc[k] -= drop
                cur -= drop

    return alloc


In [13]:
# ==========================================
# Base model fit / OOF
# ==========================================
def fit_best_model(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    groups_train: np.ndarray,
    model_type: str,
    n_splits_inner: int = 3,
):
    n_unique_groups = int(pd.Series(groups_train).nunique())
    inner_splits = min(n_splits_inner, n_unique_groups)

    if inner_splits < 2:
        raise ValueError(
            f"Not enough unique groups for inner CV: "
            f"n_unique_groups={n_unique_groups}, requested={n_splits_inner}"
        )

    inner_cv = GroupKFold(n_splits=inner_splits)

    pipe = make_pipeline(X_train, model_type=model_type)
    grid = make_param_grid(model_type=model_type)

    gs = GridSearchCV(
        estimator=pipe,
        param_grid=grid,
        scoring="r2",
        cv=inner_cv,
        n_jobs=8,
        refit=True,
    )
    gs.fit(X_train, y_train, groups=groups_train)
    return gs.best_estimator_, gs.best_params_, gs.best_score_

def get_inner_oof_predictions(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    groups_train: np.ndarray,
    model_type: str,
    best_params: dict,
    n_splits_inner: int = 3,
) -> np.ndarray:
    n_unique_groups = int(pd.Series(groups_train).nunique())
    inner_splits = min(n_splits_inner, n_unique_groups)

    if inner_splits < 2:
        return np.full(len(X_train), np.nan, dtype=float)

    inner_cv = GroupKFold(n_splits=inner_splits)
    oof = np.zeros(len(X_train), dtype=float)

    for tr_idx, va_idx in inner_cv.split(X_train, y_train, groups=groups_train):
        X_tr = X_train.iloc[tr_idx]
        y_tr = y_train.iloc[tr_idx]
        X_va = X_train.iloc[va_idx]

        pipe = make_pipeline(X_tr, model_type=model_type)
        pipe.set_params(**best_params)
        pipe.fit(X_tr, y_tr)
        oof[va_idx] = pipe.predict(X_va)

    return oof


def get_selected_feature_names(fitted_pipeline: Pipeline) -> list[str]:
    preprocess = fitted_pipeline.named_steps["preprocess"]
    selector = fitted_pipeline.named_steps["select"]
    names = preprocess.get_feature_names_out()
    mask = selector.get_support()
    return list(np.array(names)[mask])

def fit_fold_imputer(X_train: pd.DataFrame, strategy: str = "median"):
    """
    Fold-safe pre-imputer.
    숫자 컬럼만 지정된 strategy로 채우고,
    object/category 컬럼은 건드리지 않는다.
    """
    num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
    non_num_cols = [c for c in X_train.columns if c not in num_cols]

    imp_num = None
    if len(num_cols) > 0:
        imp_num = SimpleImputer(strategy=strategy)
        imp_num.fit(X_train[num_cols])

    return {
        "num_cols": num_cols,
        "non_num_cols": non_num_cols,
        "imp_num": imp_num,
        "strategy": strategy,
    }


def apply_fold_imputer(imputer_state, X: pd.DataFrame) -> pd.DataFrame:
    num_cols = imputer_state["num_cols"]
    non_num_cols = imputer_state["non_num_cols"]
    imp_num = imputer_state["imp_num"]

    parts = []

    if len(num_cols) > 0:
        X_num = pd.DataFrame(
            imp_num.transform(X[num_cols]),
            index=X.index,
            columns=num_cols,
        )
        parts.append(X_num)

    if len(non_num_cols) > 0:
        X_non_num = X[non_num_cols].copy()
        parts.append(X_non_num)

    if len(parts) == 0:
        return X.copy()

    X_out = pd.concat(parts, axis=1)
    X_out = X_out[X.columns]  # 원래 컬럼 순서 복원
    return X_out

In [14]:
def apply_feature_weights_to_matrix(
    X_t: pd.DataFrame,
    weight_df: pd.DataFrame,
) -> pd.DataFrame:
    w = pd.Series(1.0, index=X_t.columns, dtype=float)

    if weight_df is not None and len(weight_df) > 0:
        w_update = weight_df.set_index("feature")["weight"]
        common = X_t.columns.intersection(w_update.index)
        w.loc[common] = w_update.loc[common]

    return X_t.mul(w, axis=1)

In [15]:
def fit_preprocess_only(X_train: pd.DataFrame):
    preprocess = make_preprocess(X_train)
    preprocess.fit(X_train)
    return preprocess


def transform_with_preprocess(preprocess, X: pd.DataFrame) -> pd.DataFrame:
    
    Xt = preprocess.transform(X)
    feat_names = preprocess.get_feature_names_out().tolist()

    if sparse.issparse(Xt):
        Xt = Xt.toarray()

    Xt = np.asarray(Xt)

    if Xt.ndim == 1:
        Xt = Xt.reshape(-1, 1)

    if Xt.shape[1] != len(feat_names):
        raise ValueError(
            f"transform_with_preprocess column mismatch: "
            f"Xt.shape={Xt.shape}, len(feat_names)={len(feat_names)}"
        )

    return pd.DataFrame(Xt, index=X.index, columns=feat_names)


def allocate_blockwise_k(block_map, total_k, min_per_block=10):
    omics_names = ["PROT", "MET", "RNA"]
    sizes = {k: len(block_map.get(k, [])) for k in omics_names}
    active = [k for k in omics_names if sizes[k] > 0]

    if total_k is None or len(active) == 0:
        return {k: sizes[k] for k in omics_names}

    total_k = int(total_k)
    total_available = sum(sizes[k] for k in active)
    total_k = min(total_k, total_available)

    alloc = {k: 0 for k in omics_names}

    for k in active:
        alloc[k] = min(min_per_block, sizes[k])

    used = sum(alloc.values())

    if used > total_k:
        alloc = {k: 0 for k in omics_names}
        base = max(1, total_k // len(active))
        rem = total_k

        for k in active:
            take = min(base, sizes[k], rem)
            alloc[k] = take
            rem -= take

        for k in sorted(active, key=lambda x: sizes[x], reverse=True):
            if rem <= 0:
                break
            extra = min(sizes[k] - alloc[k], rem)
            alloc[k] += extra
            rem -= extra

        return alloc

    remaining = total_k - used
    if remaining > 0:
        size_sum = sum(sizes[k] for k in active)
        for k in active:
            extra = int(round(remaining * sizes[k] / size_sum))
            alloc[k] += extra

        for k in active:
            alloc[k] = min(alloc[k], sizes[k])

        cur = sum(alloc.values())

        if cur < total_k:
            for k in sorted(active, key=lambda x: sizes[x] - alloc[x], reverse=True):
                if cur >= total_k:
                    break
                room = sizes[k] - alloc[k]
                take = min(room, total_k - cur)
                alloc[k] += take
                cur += take

        elif cur > total_k:
            for k in sorted(active, key=lambda x: alloc[x], reverse=True):
                if cur <= total_k:
                    break
                drop = min(alloc[k], cur - total_k)
                alloc[k] -= drop
                cur -= drop

    return alloc

In [16]:

def build_feature_weights(
    feature_score_df: pd.DataFrame,
    mode: str = "none",
    top_k: int = 50,
    high_weight: float | None = None,
    low_weight: float | None = None,
    min_weight: float | None = None,
    max_weight: float | None = None,
):
    """
    Build feature-wise weights from train-side feature scores only.

    Safe default:
    - none: all weights = 1.0
    - soft: feature_score를 1.0~1.5 범위로 부드럽게 scaling
    - topk: optional, but low weight is also >=1.0 to avoid aggressively killing features

    NOTE:
    feature_score_df must be computed inside the current train fold only.
    """
    df = feature_score_df.copy()

    if len(df) == 0:
        return pd.DataFrame(columns=[
            "feature", "weight", "feature_weight_mode",
            "feature_weight_min", "feature_weight_max"
        ])

    high_weight = FEATURE_WEIGHT_TOPK_HIGH if high_weight is None else high_weight
    low_weight = FEATURE_WEIGHT_TOPK_LOW if low_weight is None else low_weight
    min_weight = FEATURE_WEIGHT_MIN if min_weight is None else min_weight
    max_weight = FEATURE_WEIGHT_MAX if max_weight is None else max_weight

    # enforce safe range
    low_weight = max(float(low_weight), float(FEATURE_WEIGHT_DEFAULT))
    min_weight = max(float(min_weight), float(FEATURE_WEIGHT_DEFAULT))
    high_weight = min(float(high_weight), float(max_weight))
    max_weight = max(float(max_weight), float(min_weight))

    if "feature_score" not in df.columns:
        df["feature_score"] = 0.0

    if mode == "none":
        df["weight"] = float(FEATURE_WEIGHT_DEFAULT)

    elif mode == "topk":
        df["weight"] = float(low_weight)
        top_feats = df.head(int(min(top_k, len(df))))["feature"].tolist()
        df.loc[df["feature"].isin(top_feats), "weight"] = float(high_weight)

    elif mode == "soft":
        s = pd.to_numeric(df["feature_score"], errors="coerce").fillna(0.0).values.astype(float)
        s_min, s_max = np.nanmin(s), np.nanmax(s)

        if np.isclose(s_min, s_max):
            scaled = np.ones_like(s)
        else:
            scaled = (s - s_min) / (s_max - s_min)

        df["weight"] = float(min_weight) + scaled * (float(max_weight) - float(min_weight))

    else:
        raise ValueError(f"Unknown weighting mode: {mode}")

    df["weight"] = pd.to_numeric(df["weight"], errors="coerce").fillna(FEATURE_WEIGHT_DEFAULT)
    df["weight"] = df["weight"].clip(lower=FEATURE_WEIGHT_DEFAULT, upper=FEATURE_WEIGHT_MAX)

    df["feature_weight_mode"] = mode
    df["feature_weight_min"] = FEATURE_WEIGHT_MIN
    df["feature_weight_max"] = FEATURE_WEIGHT_MAX

    return df[[
        "feature", "weight", "feature_weight_mode",
        "feature_weight_min", "feature_weight_max"
    ]].copy()


In [17]:

def compute_univariate_feature_scores(
    X_train_use: pd.DataFrame,
    y_train: pd.Series,
    feature_selection_mode: str = "f_regression_topk",
) -> pd.Series:
    """
    Fold-safe univariate score computed using the current train split only.

    Modes:
    - f_regression_topk: sklearn f_regression score, closest to paper-style univariate association
    - corr_topk: absolute Pearson correlation with DeltaTMS
    - hybrid_topk: average rank of f_regression and absolute correlation
    """
    if X_train_use.shape[1] == 0:
        return pd.Series(dtype=float)

    if feature_selection_mode == "f_regression_topk":
        scores, _ = f_regression(X_train_use, y_train)
        score_s = pd.Series(scores, index=X_train_use.columns)

    elif feature_selection_mode == "corr_topk":
        y_num = pd.to_numeric(y_train, errors="coerce")
        vals = {}
        for c in X_train_use.columns:
            x = pd.to_numeric(X_train_use[c], errors="coerce")
            common = x.notna() & y_num.notna()
            if common.sum() < 3:
                vals[c] = 0.0
            else:
                vals[c] = abs(float(np.corrcoef(x.loc[common], y_num.loc[common])[0, 1]))
        score_s = pd.Series(vals)

    elif feature_selection_mode == "hybrid_topk":
        f_scores, _ = f_regression(X_train_use, y_train)
        f_s = (
            pd.Series(f_scores, index=X_train_use.columns)
            .replace([np.inf, -np.inf], np.nan)
            .fillna(0.0)
        )
        y_num = pd.to_numeric(y_train, errors="coerce")
        corr_vals = {}
        for c in X_train_use.columns:
            x = pd.to_numeric(X_train_use[c], errors="coerce")
            common = x.notna() & y_num.notna()
            if common.sum() < 3:
                corr_vals[c] = 0.0
            else:
                corr_vals[c] = abs(float(np.corrcoef(x.loc[common], y_num.loc[common])[0, 1]))
        c_s = pd.Series(corr_vals).replace([np.inf, -np.inf], np.nan).fillna(0.0)

        # Convert to percentile ranks so the two score scales are comparable.
        score_s = 0.5 * f_s.rank(pct=True) + 0.5 * c_s.rank(pct=True)

    else:
        raise ValueError(f"Unknown feature_selection_mode: {feature_selection_mode}")

    return (
        score_s
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .astype(float)
    )


def select_features_manual(
    X_train_t: pd.DataFrame,
    y_train: pd.Series,
    X_valid_t: pd.DataFrame,
    k: int | None = None,
    feature_selection_mode: str = "f_regression_topk",
):
    if X_train_t.shape[1] == 0:
        return X_train_t.copy(), X_valid_t.copy(), []

    # variance 0 제거
    valid_cols = X_train_t.columns[X_train_t.var(axis=0) > 0].tolist()
    if len(valid_cols) == 0:
        return (
            X_train_t.iloc[:, :0].copy(),
            X_valid_t.iloc[:, :0].copy(),
            [],
        )

    X_train_use = X_train_t[valid_cols].copy()
    X_valid_use = X_valid_t[valid_cols].copy()

    # 전체 유지
    if k is None or k >= X_train_use.shape[1]:
        return X_train_use.copy(), X_valid_use.copy(), X_train_use.columns.tolist()

    # global fallback helper
    def _global_select():
        score_s = compute_univariate_feature_scores(
            X_train_use=X_train_use,
            y_train=y_train,
            feature_selection_mode=feature_selection_mode,
        )
        selected_global = score_s.sort_values(ascending=False).head(k).index.tolist()
        return (
            X_train_use[selected_global].copy(),
            X_valid_use[selected_global].copy(),
            selected_global,
        )

    if not USE_BLOCKWISE_EF:
        return _global_select()

    block_map = split_omics_blocks_transformed(
        X_train_use.columns,
        protected_keep_cols=LIGHT_CLIN_COLS,
    )

    print(
        "[DEBUG][BLOCKWISE] counts =",
        {
            "PROT": len(block_map["PROT"]),
            "MET": len(block_map["MET"]),
            "RNA": len(block_map["RNA"]),
            "CLIN": len(block_map["CLIN"]),
        }
    )

    # block를 하나도 못 찾으면 global fallback
    n_omics_found = len(block_map["PROT"]) + len(block_map["MET"]) + len(block_map["RNA"])
    if n_omics_found == 0:
        print("[WARN][BLOCKWISE] no omics block found from transformed columns -> fallback to global selection")
        return _global_select()

    alloc = allocate_blockwise_k(
        block_map=block_map,
        total_k=k,
        min_per_block=BLOCKWISE_MIN_PER_BLOCK,
    )

    selected = []

    # clinical transformed columns 유지
    clin_cols = [c for c in block_map["CLIN"] if c in X_train_use.columns]
    selected.extend(clin_cols)

    for block_name in ["PROT", "MET", "RNA"]:
        cols = [c for c in block_map[block_name] if c in X_train_use.columns]
        if len(cols) == 0:
            continue

        block_k = alloc.get(block_name, 0)
        if block_k <= 0:
            continue

        X_tr_block = X_train_use[cols].copy()
        score_s = compute_univariate_feature_scores(
            X_train_use=X_tr_block,
            y_train=y_train,
            feature_selection_mode=feature_selection_mode,
        )

        chosen = score_s.sort_values(ascending=False).head(block_k).index.tolist()
        selected.extend(chosen)

    selected = list(dict.fromkeys(selected))
    selected = [c for c in selected if c in X_train_use.columns]

    if len(selected) == 0:
        return _global_select()

    return (
        X_train_use[selected].copy(),
        X_valid_use[selected].copy(),
        selected,
    )


In [18]:
def collect_feature_scores_inner_cv(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    groups_train: np.ndarray,
    model_type: str,
    model_params: dict,
    n_splits_inner: int = 3,
    prefilter_k: int | None = None,
    feature_selection_mode: str = "f_regression_topk",
):
    if X_train.shape[1] == 0:
        return (
            pd.DataFrame(columns=["feature", "selection_freq", "mean_abs_coef", "feature_score"]),
            pd.DataFrame(),
        )

    inner_splits_list, inner_cv_type = make_inner_cv_splits(
        X=X_train,
        y=y_train,
        groups=groups_train,
        n_splits=n_splits_inner,
    )

    inner_splits = len(inner_splits_list)

    if inner_splits < 2:
        return (
            pd.DataFrame(columns=["feature", "selection_freq", "mean_abs_coef", "feature_score"]),
            pd.DataFrame(),
        )

    freq_counter = {}
    coef_sum = {}
    coef_count = {}
    fold_rows = []

    for fold, (tr_idx, va_idx) in enumerate(inner_splits_list, start=1):
        X_tr_raw = X_train.iloc[tr_idx].copy()
        y_tr_raw = y_train.iloc[tr_idx].copy()
        X_va_raw = X_train.iloc[va_idx].copy()
        y_va = y_train.iloc[va_idx].copy()

        try:
            X_tr, y_tr, X_va, prep_info = prepare_train_valid_fold(
                X_train=X_tr_raw,
                y_train=y_tr_raw,
                X_valid=X_va_raw,
                protected_keep_cols=LIGHT_CLIN_COLS,
                min_obs_frac=MIN_OBS_FRAC,
                fallback_thresholds=SPARSE_FALLBACK_THRESHOLDS,
                min_omics_features=MIN_OMICS_FEATURES_AFTER_FILTER,
            )
        except Exception:
            continue

        preprocess = fit_preprocess_only(X_tr)
        X_tr_t = transform_with_preprocess(preprocess, X_tr)
        X_va_t = transform_with_preprocess(preprocess, X_va)

        X_tr_s, X_va_s, selected_feats = select_features_manual(
            X_train_t=X_tr_t,
            y_train=y_tr,
            X_valid_t=X_va_t,
            k=prefilter_k,
            feature_selection_mode=feature_selection_mode,
        )

        block_map_selected = split_omics_blocks_transformed(X_tr_s.columns, protected_keep_cols=LIGHT_CLIN_COLS)

        print(
            f"[INNER-SELECT] fold={fold} model={model_type} "
            f"| prefilter_k={prefilter_k} "
            f"| feature_selection={feature_selection_mode} "
            f"| before={X_tr_t.shape[1]} after={X_tr_s.shape[1]} "
            f"| prot={len(block_map_selected['PROT'])} "
            f"| met={len(block_map_selected['MET'])} "
            f"| rna={len(block_map_selected['RNA'])} "
            f"| clin={len(block_map_selected['CLIN'])}"
        )

        if X_tr_s.shape[1] == 0:
            continue

        model = make_model(model_type)
        try:
            model.set_params(**sanitize_model_params(model_type, model_params, X_tr_s))
            model.fit(X_tr_s, y_tr)
        except Exception as e:
            print(
                f"[INNER-SKIP] fold={fold} model={model_type} "
                f"| reason={type(e).__name__}: {e}"
            )
            continue

        pred_va = model.predict(X_va_s)
        fold_rows.append({
            "fold": fold,
            "inner_cv_type": inner_cv_type,
            "feature_selection_mode": feature_selection_mode,
            "valid_r2": r2_score(y_va, pred_va),
            "valid_mae": mean_absolute_error(y_va, pred_va),
            "n_train_after_row_filter": len(X_tr),
            "n_valid": len(X_va),
            "n_features_after_sparse": X_tr.shape[1],
            "n_selected_prefilter": X_tr_s.shape[1],
        })

        for f in selected_feats:
            freq_counter[f] = freq_counter.get(f, 0) + 1

        if hasattr(model, "coef_"):
            coef_abs = np.abs(np.ravel(model.coef_))
        elif hasattr(model, "feature_importances_"):
            coef_abs = np.abs(np.ravel(model.feature_importances_))
        else:
            coef_abs = np.ones(len(X_tr_s.columns), dtype=float)

        if len(coef_abs) == len(X_tr_s.columns):
            for f, c in zip(X_tr_s.columns, coef_abs):
                coef_sum[f] = coef_sum.get(f, 0.0) + float(c)
                coef_count[f] = coef_count.get(f, 0) + 1

    all_feats = sorted(set(list(freq_counter.keys()) + list(coef_sum.keys())))
    rows = []
    for f in all_feats:
        freq = freq_counter.get(f, 0) / inner_splits
        mean_coef = coef_sum.get(f, 0.0) / max(coef_count.get(f, 1), 1)
        rows.append({
            "feature": f,
            "selection_freq": freq,
            "mean_abs_coef": mean_coef,
        })

    score_df = pd.DataFrame(rows)
    if len(score_df) == 0:
        return (
            pd.DataFrame(columns=["feature", "selection_freq", "mean_abs_coef", "feature_score"]),
            pd.DataFrame(fold_rows),
        )

    score_df["freq_z"] = score_df["selection_freq"] / max(score_df["selection_freq"].max(), 1e-12)
    score_df["coef_z"] = score_df["mean_abs_coef"] / max(score_df["mean_abs_coef"].max(), 1e-12)
    score_df["feature_score"] = 0.6 * score_df["freq_z"] + 0.4 * score_df["coef_z"]
    score_df = score_df.sort_values("feature_score", ascending=False).reset_index(drop=True)

    return score_df, pd.DataFrame(fold_rows)


In [19]:

# ==========================================
# Nested-CV weighted model search
# ==========================================

def make_weighted_search_grid(model_type: str, tp: int | None = None, tissue: str | None = None) -> list[dict]:
    """
    v13 adaptive nested-CV grid.

    The EF space is tuned by tissue/timepoint routing; grid size still depends on signal strength:
      - strong: wider grid for promising windows, especially CSF 72/96 and serum 96
      - middle: moderate linear/PLS/SVR search
      - weak: small conservative linear search only
    """
    tp = int(tp) if tp is not None else None
    tissue = str(tissue).lower() if tissue is not None else "unknown"
    tuning_group = get_ef_tuning_group(tissue, tp) if tp is not None and tissue != "unknown" else "middle"
    is_late = tp is not None and tp >= LATE_TP_START
    fs_modes = FEATURE_SELECTION_MODES_TO_RUN
    weight_modes = FEATURE_WEIGHT_MODES

    rows = []

    if tuning_group == "strong":
        ridge_alphas = [1.0, 5.0, 10.0, 50.0, 100.0, 200.0]
        ridge_topks = [30, 40, 60]
        enet_alphas = [0.01, 0.05, 0.1, 0.5, 1.0]
        enet_l1s = [0.05, 0.1, 0.2, 0.4]
        enet_topks = [20, 30, 40]
        pls_components = [1, 2, 3]
        pls_topks = [20, 30, 40]
        svr_Cs = [0.1, 0.3, 1.0, 3.0]
        svr_eps = [0.05, 0.1, 0.2]
        svr_topks = [20, 30, 40]
        gbr_estimators = [50, 80, 120]
        gbr_lrs = [0.03, 0.05]
        gbr_depths = [1, 2]
        gbr_leafs = [3, 5]
        gbr_subsamples = [0.8, 1.0]
        gbr_topks = [20, 30]
    elif tuning_group == "middle":
        ridge_alphas = [5.0, 10.0, 50.0, 100.0, 200.0]
        ridge_topks = [30, 40, 60]
        enet_alphas = [0.05, 0.1, 0.5, 1.0]
        enet_l1s = [0.05, 0.1, 0.2]
        enet_topks = [20, 30, 40]
        pls_components = [1, 2]
        pls_topks = [20, 40]
        svr_Cs = [0.3, 1.0]
        svr_eps = [0.1, 0.2]
        svr_topks = [30, 40]
        gbr_estimators = [50, 80]
        gbr_lrs = [0.03]
        gbr_depths = [1]
        gbr_leafs = [3, 5]
        gbr_subsamples = [0.8]
        gbr_topks = [20]
    else:
        ridge_alphas = [10.0, 50.0, 100.0, 200.0]
        ridge_topks = [30, 40]
        enet_alphas = [0.1, 0.5, 1.0]
        enet_l1s = [0.05, 0.1]
        enet_topks = [20, 30]
        pls_components = [1]
        pls_topks = [20]
        svr_Cs = [0.3]
        svr_eps = [0.1]
        svr_topks = [30]
        gbr_estimators = [50]
        gbr_lrs = [0.03]
        gbr_depths = [1]
        gbr_leafs = [5]
        gbr_subsamples = [0.8]
        gbr_topks = [20]

    if model_type == "ridge":
        for alpha in ridge_alphas:
            for top_k in ridge_topks:
                for weight_mode in weight_modes:
                    for fs_mode in fs_modes:
                        rows.append({
                            "model_type": model_type,
                            "model_params": {"alpha": alpha},
                            "prefilter_k": max(top_k, 60 if not is_late else 40),
                            "top_k": top_k,
                            "weight_mode": weight_mode,
                            "feature_selection_mode": fs_mode,
                            "tuning_group": tuning_group,
                        })

    elif model_type == "elasticnet":
        for alpha in enet_alphas:
            for l1_ratio in enet_l1s:
                for top_k in enet_topks:
                    for weight_mode in weight_modes:
                        for fs_mode in fs_modes:
                            rows.append({
                                "model_type": model_type,
                                "model_params": {"alpha": alpha, "l1_ratio": l1_ratio},
                                "prefilter_k": max(top_k, 40),
                                "top_k": top_k,
                                "weight_mode": weight_mode,
                                "feature_selection_mode": fs_mode,
                                "tuning_group": tuning_group,
                            })

    elif model_type == "pls":
        for n_components in pls_components:
            for top_k in pls_topks:
                for fs_mode in fs_modes:
                    rows.append({
                        "model_type": model_type,
                        "model_params": {"n_components": n_components},
                        "prefilter_k": max(top_k, 40),
                        "top_k": top_k,
                        "weight_mode": "none",
                        "feature_selection_mode": fs_mode,
                        "tuning_group": tuning_group,
                    })

    elif model_type == "svr_linear":
        for C in svr_Cs:
            for epsilon in svr_eps:
                for top_k in svr_topks:
                    for weight_mode in weight_modes:
                        for fs_mode in fs_modes:
                            rows.append({
                                "model_type": model_type,
                                "model_params": {"C": C, "epsilon": epsilon},
                                "prefilter_k": max(top_k, 40),
                                "top_k": top_k,
                                "weight_mode": weight_mode,
                                "feature_selection_mode": fs_mode,
                                "tuning_group": tuning_group,
                            })

    elif model_type == "gbr":
        for n_estimators in gbr_estimators:
            for learning_rate in gbr_lrs:
                for max_depth in gbr_depths:
                    for min_samples_leaf in gbr_leafs:
                        for subsample in gbr_subsamples:
                            for top_k in gbr_topks:
                                for fs_mode in fs_modes:
                                    rows.append({
                                        "model_type": model_type,
                                        "model_params": {
                                            "n_estimators": n_estimators,
                                            "learning_rate": learning_rate,
                                            "max_depth": max_depth,
                                            "min_samples_leaf": min_samples_leaf,
                                            "subsample": subsample,
                                        },
                                        "prefilter_k": max(top_k, 30),
                                        "top_k": top_k,
                                        "weight_mode": "none",
                                        "feature_selection_mode": fs_mode,
                                        "tuning_group": tuning_group,
                                    })

    else:
        raise ValueError(f"Unsupported model_type for v13 nested CV grid: {model_type}")

    return rows

def _cap_model_params_for_data(model_type: str, model_params: dict, n_samples: int, n_features: int) -> dict:
    """Make model params safe for the current fold, especially PLS n_components."""
    out = dict(model_params)
    if model_type == "pls":
        max_components = max(1, min(int(n_samples) - 1, int(n_features)))
        requested = int(out.get("n_components", 2))
        out["n_components"] = int(max(1, min(requested, max_components)))
    return out


def _fit_weighted_config(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    config: dict,
):
    """
    Fit one weighted EF config on a given training split.
    Returns all fold-local objects needed for prediction and logging.
    """
    model_type = config["model_type"]
    prefilter_k = int(config["prefilter_k"])
    top_k = int(config["top_k"])
    weight_mode = config["weight_mode"]
    feature_selection_mode = config["feature_selection_mode"]

    preprocess = fit_preprocess_only(X_train)
    X_train_t = transform_with_preprocess(preprocess, X_train)

    X_train_s, _, selected_features = select_features_manual(
        X_train_t=X_train_t,
        y_train=y_train,
        X_valid_t=X_train_t,
        k=prefilter_k,
        feature_selection_mode=feature_selection_mode,
    )

    selected_features = list(dict.fromkeys(selected_features))
    if top_k is not None and len(selected_features) > top_k:
        score_tmp = compute_univariate_feature_scores(
            X_train_use=X_train_t[selected_features].copy(),
            y_train=y_train,
            feature_selection_mode=feature_selection_mode,
        )
        selected_features = score_tmp.sort_values(ascending=False).head(top_k).index.tolist()
        X_train_s = X_train_t[selected_features].copy()

    if len(selected_features) == 0:
        raise ValueError(f"No selected features for config={config}")

    score_s = compute_univariate_feature_scores(
        X_train_use=X_train_t[selected_features].copy(),
        y_train=y_train,
        feature_selection_mode=feature_selection_mode,
    )
    feature_score_df = pd.DataFrame({
        "feature": selected_features,
        "selection_freq": 1.0,
        "mean_abs_coef": score_s.reindex(selected_features).fillna(0.0).values,
        "feature_score": score_s.reindex(selected_features).fillna(0.0).values,
    })
    max_score = max(float(feature_score_df["feature_score"].max()), 1e-12)
    feature_score_df["feature_score"] = feature_score_df["feature_score"] / max_score
    feature_score_df = feature_score_df.sort_values("feature_score", ascending=False).reset_index(drop=True)

    weight_df = build_feature_weights(
        feature_score_df=feature_score_df,
        mode=weight_mode,
        top_k=top_k,
        min_weight=FEATURE_WEIGHT_MIN,
        max_weight=FEATURE_WEIGHT_MAX,
    )

    X_train_w = apply_feature_weights_to_matrix(X_train_s, weight_df)

    model_params = _cap_model_params_for_data(
        model_type=model_type,
        model_params=config["model_params"],
        n_samples=X_train_w.shape[0],
        n_features=X_train_w.shape[1],
    )
    model = make_model(model_type)
    model.set_params(**model_params)
    model.fit(X_train_w, y_train)

    return {
        "preprocess": preprocess,
        "selected_features": selected_features,
        "feature_score_df": feature_score_df,
        "feature_weight_df": weight_df,
        "fitted_model": model,
        "model_params": model_params,
        "X_train_w": X_train_w,
    }


def _predict_weighted_fit(fit_obj: dict, X: pd.DataFrame) -> pd.Series:
    preprocess = fit_obj["preprocess"]
    selected_features = fit_obj["selected_features"]
    weight_df = fit_obj["feature_weight_df"]
    model = fit_obj["fitted_model"]

    X_t = transform_with_preprocess(preprocess, X)
    X_s = X_t[selected_features].copy()
    w_sub = weight_df[weight_df["feature"].isin(selected_features)].copy()
    X_w = apply_feature_weights_to_matrix(X_s, w_sub)
    return pd.Series(model.predict(X_w), index=X.index, dtype=float)


def fit_best_weighted_model(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    groups_train: np.ndarray,
    candidate_models: list[str],
    n_splits_inner: int = 3,
    tp: int | None = None,
    tissue: str | None = None,
):
    """
    v10 full nested-CV weighted model search.

    Outer fold is handled by run_early_fusion_cv(). Inside the current outer training fold,
    this function runs GroupKFold inner CV to select the best config for the requested model.

    Since run_one_ef_experiment passes one model_type per job, candidate_models normally has length 1.
    Keeping the list interface preserves compatibility with older notebook code.
    """
    if len(candidate_models) < 1:
        raise ValueError("candidate_models must contain at least one model type")

    inner_splits_list, inner_cv_type = make_inner_cv_splits(
        X=X_train,
        y=y_train,
        groups=groups_train,
        n_splits=n_splits_inner,
    )
    inner_splits = len(inner_splits_list)
    if inner_splits < 2:
        n_unique_groups = int(pd.Series(groups_train).nunique())
        raise ValueError(
            f"Not enough data for inner CV: n_unique_groups={n_unique_groups}, requested={n_splits_inner}"
        )

    search_rows = []
    best_record = None

    for model_type in candidate_models:
        for cfg_idx, config in enumerate(make_weighted_search_grid(model_type=model_type, tp=tp, tissue=tissue), start=1):
            fold_scores = []
            fold_maes = []
            fold_train_scores = []
            fold_status = []

            for inner_fold, (tr_idx, va_idx) in enumerate(inner_splits_list, start=1):
                X_in_tr = X_train.iloc[tr_idx].copy()
                y_in_tr = y_train.iloc[tr_idx].copy()
                X_in_va = X_train.iloc[va_idx].copy()
                y_in_va = y_train.iloc[va_idx].copy()

                try:
                    fit_obj = _fit_weighted_config(X_in_tr, y_in_tr, config)
                    pred_va = _predict_weighted_fit(fit_obj, X_in_va)
                    pred_tr = _predict_weighted_fit(fit_obj, X_in_tr)

                    valid_r2 = r2_score(y_in_va, pred_va) if len(y_in_va) > 1 else np.nan
                    valid_mae = mean_absolute_error(y_in_va, pred_va) if len(y_in_va) > 0 else np.nan
                    train_r2 = r2_score(y_in_tr, pred_tr) if len(y_in_tr) > 1 else np.nan

                    fold_scores.append(valid_r2)
                    fold_maes.append(valid_mae)
                    fold_train_scores.append(train_r2)
                    fold_status.append("ok")
                except Exception as e:
                    fold_scores.append(np.nan)
                    fold_maes.append(np.nan)
                    fold_train_scores.append(np.nan)
                    fold_status.append(f"failed: {type(e).__name__}: {e}")

            score_arr = np.asarray(fold_scores, dtype=float)
            mae_arr = np.asarray(fold_maes, dtype=float)
            train_arr = np.asarray(fold_train_scores, dtype=float)
            ok_mask = np.isfinite(score_arr)

            mean_inner_r2 = float(np.nanmean(score_arr)) if ok_mask.any() else np.nan
            median_inner_r2 = float(np.nanmedian(score_arr)) if ok_mask.any() else np.nan
            std_inner_r2 = float(np.nanstd(score_arr, ddof=1)) if ok_mask.sum() > 1 else 0.0
            min_inner_r2 = float(np.nanmin(score_arr)) if ok_mask.any() else np.nan
            mean_inner_mae = float(np.nanmean(mae_arr)) if np.isfinite(mae_arr).any() else np.nan
            mean_train_r2 = float(np.nanmean(train_arr)) if np.isfinite(train_arr).any() else np.nan

            # Stability-aware objective. This avoids selecting configs that win only by one lucky fold.
            # v10 slightly strengthens the worst-fold term because v9 had high train R2 but weak OOF R2.
            selection_score = mean_inner_r2 - 0.50 * std_inner_r2 + 0.20 * min_inner_r2
            if not np.isfinite(selection_score):
                selection_score = -np.inf

            row = {
                "cv_search_mode": CV_SEARCH_MODE,
                "inner_cv_type": inner_cv_type,
                "tuning_group": config.get("tuning_group", ""),
                "model_type": model_type,
                "config_idx": cfg_idx,
                "tp": tp,
                "selection_score": selection_score,
                "mean_inner_r2": mean_inner_r2,
                "median_inner_r2": median_inner_r2,
                "std_inner_r2": std_inner_r2,
                "min_inner_r2": min_inner_r2,
                "mean_inner_mae": mean_inner_mae,
                "mean_inner_train_r2": mean_train_r2,
                "n_ok_inner_folds": int(ok_mask.sum()),
                "n_inner_folds": int(inner_splits),
                "prefilter_k": config["prefilter_k"],
                "top_k": config["top_k"],
                "weight_mode": config["weight_mode"],
                "feature_selection_mode": config["feature_selection_mode"],
                **{f"model_param__{k}": v for k, v in config["model_params"].items()},
            }
            for j, val in enumerate(score_arr, start=1):
                row[f"inner_fold{j}_r2"] = val
            for j, val in enumerate(mae_arr, start=1):
                row[f"inner_fold{j}_mae"] = val
            search_rows.append(row)

            if best_record is None or selection_score > best_record["selection_score"]:
                best_record = {
                    "selection_score": selection_score,
                    "config": config,
                    "row": row,
                }

    search_df = pd.DataFrame(search_rows)
    if best_record is None or not np.isfinite(best_record["selection_score"]):
        raise ValueError("No valid inner-CV config was found")

    best_config = best_record["config"]

    # Refit the selected config on the full current outer-training fold.
    final_fit = _fit_weighted_config(X_train, y_train, best_config)
    pred_train = _predict_weighted_fit(final_fit, X_train)
    train_r2 = r2_score(y_train, pred_train) if len(y_train) > 1 else np.nan
    train_mae = mean_absolute_error(y_train, pred_train) if len(y_train) > 0 else np.nan

    selected_features = final_fit["selected_features"]
    feature_score_df = final_fit["feature_score_df"].copy()

    # Add model coefficient/importances if available for interpretability.
    coef_score_df = build_local_feature_score_df(final_fit["fitted_model"], selected_features)
    if len(coef_score_df) > 0:
        feature_score_df = feature_score_df.merge(
            coef_score_df[["feature", "mean_abs_coef"]].rename(columns={"mean_abs_coef": "model_abs_coef"}),
            on="feature",
            how="left",
        )

    weight_df = final_fit["feature_weight_df"].copy()
    best_params = {
        "model_params": final_fit["model_params"],
        "prefilter_k": best_config["prefilter_k"],
        "top_k": best_config["top_k"],
        "weight_mode": best_config["weight_mode"],
        "feature_selection_mode": best_config["feature_selection_mode"],
        **{f"model_param__{k}": v for k, v in final_fit["model_params"].items()},
    }

    return {
        "best_model_type": best_config["model_type"],
        "best_params": best_params,
        "best_score": float(best_record["row"]["selection_score"]),
        "best_mean_inner_r2": float(best_record["row"]["mean_inner_r2"]),
        "best_std_inner_r2": float(best_record["row"]["std_inner_r2"]),
        "best_median_inner_r2": float(best_record["row"]["median_inner_r2"]),
        "best_min_inner_r2": float(best_record["row"]["min_inner_r2"]),
        "best_train_r2_outer_fold": train_r2,
        "best_train_mae_outer_fold": train_mae,
        "inner_cv_type": inner_cv_type,
        "search_df": search_df,
        "feature_score_df": feature_score_df,
        "feature_weight_df": weight_df,
        "preprocess": final_fit["preprocess"],
        "selected_features": selected_features,
        "fitted_model": final_fit["fitted_model"],
        "feature_weight_mode": best_config["weight_mode"],
        "feature_selection_mode": best_config["feature_selection_mode"],
        "mean_feature_weight": float(weight_df["weight"].mean()) if len(weight_df) > 0 else np.nan,
        "max_feature_weight": float(weight_df["weight"].max()) if len(weight_df) > 0 else np.nan,
        "n_weighted_features": int((weight_df["weight"] != FEATURE_WEIGHT_DEFAULT).sum()) if len(weight_df) > 0 else 0,
    }



In [20]:
def make_regression_strata(
    y: pd.Series,
    n_bins: int,
    n_splits: int,
) -> pd.Series:
    """
    Quantile-based strata for regression CV.

    목적:
    - small-n regression에서 fold별 DeltaTMS 분포가 크게 달라지는 문제를 줄임
    - LF / single-omics와 같은 StratifiedKFold_y_bins 기준으로 EF를 비교 가능하게 만듦
    """
    y_num = pd.to_numeric(y, errors="coerce")

    if y_num.notna().sum() < n_splits:
        return pd.Series(index=y.index, data=0).astype(int)

    max_bins = min(n_bins, int(y_num.nunique()))

    for bins in range(max_bins, 1, -1):
        try:
            y_bin = pd.qcut(
                y_num,
                q=bins,
                labels=False,
                duplicates="drop",
            )
            y_bin = pd.Series(y_bin, index=y.index)

            counts = y_bin.value_counts(dropna=True)

            if len(counts) >= 2 and counts.min() >= n_splits:
                return y_bin.fillna(-1).astype(int)

        except Exception:
            continue

    return pd.Series(index=y.index, data=0).astype(int)


def make_outer_cv_splits(
    X: pd.DataFrame,
    y: pd.Series,
    groups: np.ndarray,
    n_splits: int,
):
    """
    Outer CV split helper.

    EF도 LF/single-omics와 공정하게 비교하기 위해
    기본적으로 DeltaTMS-stratified outer CV를 사용한다.

    fallback:
    - stratification이 불가능하면 GroupKFold 사용
    """
    if USE_STRATIFIED_OUTER_CV and X.index.is_unique:
        y_bins = make_regression_strata(
            y=y,
            n_bins=N_Y_BINS_FOR_OUTER_CV,
            n_splits=n_splits,
        )

        if y_bins.nunique(dropna=True) >= 2:
            outer_cv = StratifiedKFold(
                n_splits=n_splits,
                shuffle=True,
                random_state=OUTER_CV_RANDOM_STATE,
            )
            return list(outer_cv.split(X, y_bins)), "StratifiedKFold_y_bins"

    outer_cv = GroupKFold(n_splits=n_splits)
    return list(outer_cv.split(X, y, groups=groups)), "GroupKFold_fallback"

In [21]:
def make_inner_cv_splits(
    X: pd.DataFrame,
    y: pd.Series,
    groups: np.ndarray,
    n_splits: int,
):
    """
    Inner CV split helper.

    목적:
    - small-n regression에서 inner fold별 DeltaTMS 분포 차이를 줄임
    - hyperparameter / prefilter_k / weight_mode 선택을 더 안정화
    - patient-level index가 unique하면 y-bin stratified CV 사용
    - 불가능하면 GroupKFold로 fallback
    """
    n_unique_groups = int(pd.Series(groups).nunique())
    inner_splits = min(n_splits, n_unique_groups, len(X))

    if inner_splits < 2:
        return [], "too_few_samples"

    if X.index.is_unique:
        y_bins = make_regression_strata(
            y=y,
            n_bins=N_Y_BINS_FOR_OUTER_CV,
            n_splits=inner_splits,
        )

        if y_bins.nunique(dropna=True) >= 2:
            inner_cv = StratifiedKFold(
                n_splits=inner_splits,
                shuffle=True,
                random_state=OUTER_CV_RANDOM_STATE,
            )
            return list(inner_cv.split(X, y_bins)), "StratifiedKFold_y_bins"

    inner_cv = GroupKFold(n_splits=inner_splits)
    return list(inner_cv.split(X, y, groups=groups)), "GroupKFold_fallback"

In [22]:
def run_early_fusion_test_on_predefined_split(
    tissue: str,
    combo: str,
    tp: int,
    y_all: pd.Series,
    train_ids: list[str],
    test_ids: list[str],
    model_type: str,
    clinical_mode: str = "light",
):
    """
    Train final EF model on TRAIN_IDS only and project into held-out TEST_IDS.

    Important leakage rule:
    - Train/test split is fixed by TRAIN_IDS and TEST_IDS.
    - Sparse filtering, preprocessing, feature selection, feature weighting,
      and hyperparameter selection are computed using train patients only.
    - Test labels are used only once for final evaluation.
    """
    path = build_feature_path(
        tissue=tissue,
        combo=combo,
        tp=tp,
        use_delta=USE_DELTA_VIEW,
    )

    X = load_feature_csv(path)

    if clinical_mode == "light":
        X = merge_with_light_clinical(
            X_feat=X,
            tissue=tissue,
            combo=combo,
            tp=tp,
        )
    elif clinical_mode == "omics_only":
        X = X.copy()
    else:
        raise ValueError(f"Unknown clinical_mode: {clinical_mode}")

    X, y = align_xy(X, y_all)

    train_ids_in = [str(pid) for pid in train_ids if str(pid) in X.index]
    test_ids_in = [str(pid) for pid in test_ids if str(pid) in X.index]

    X_train_raw = X.loc[train_ids_in].copy()
    y_train_raw = y.loc[train_ids_in].copy()

    X_test_raw = X.loc[test_ids_in].copy()
    y_test = y.loc[test_ids_in].copy()

    if len(X_train_raw) < N_SPLITS_INNER:
        raise ValueError(
            f"[EF-{tissue}-{combo}-TP{tp}-{model_type}-{clinical_mode}] "
            f"not enough train patients for inner CV: n_train={len(X_train_raw)}"
        )

    if len(X_test_raw) == 0:
        raise ValueError(
            f"[EF-{tissue}-{combo}-TP{tp}-{model_type}-{clinical_mode}] "
            f"no test patients found in feature table"
        )

    # Fold-safe train-only preparation; test rows are never dropped by y.
    X_train, y_train, X_test, prep_info = prepare_train_valid_fold(
        X_train=X_train_raw,
        y_train=y_train_raw,
        X_valid=X_test_raw,
        protected_keep_cols=LIGHT_CLIN_COLS,
        min_obs_frac=MIN_OBS_FRAC,
        fallback_thresholds=SPARSE_FALLBACK_THRESHOLDS,
        min_omics_features=MIN_OMICS_FEATURES_AFTER_FILTER,
    )

    sparse_info = prep_info["sparse_info"]
    row_info = prep_info["row_info"]

    groups_train = X_train.index.to_numpy()

    search_result = fit_best_weighted_model(
        X_train=X_train,
        y_train=y_train,
        groups_train=groups_train,
        candidate_models=[model_type],
        n_splits_inner=N_SPLITS_INNER,
        tp=tp,
        tissue=tissue,
    )

    preprocess = search_result["preprocess"]
    fitted_model = search_result["fitted_model"]

    X_train_t = transform_with_preprocess(preprocess, X_train)
    X_test_t = transform_with_preprocess(preprocess, X_test)

    selected = search_result["selected_features"]
    selected = [c for c in selected if c in X_train_t.columns and c in X_test_t.columns]

    if len(selected) == 0:
        raise ValueError(
            f"No selected features remain after train/test transform alignment: "
            f"{tissue}, {combo}, tp={tp}, model={model_type}, clinical={clinical_mode}"
        )

    weight_df = search_result["feature_weight_df"]

    feature_weight_mode = str(
        search_result.get(
            "feature_weight_mode",
            search_result.get("best_params", {}).get("weight_mode", "unknown"),
        )
    )
    feature_selection_mode = str(
        search_result.get(
            "feature_selection_mode",
            search_result.get("best_params", {}).get("feature_selection_mode", "unknown"),
        )
    )

    mean_feature_weight = (
        float(weight_df["weight"].mean())
        if len(weight_df) > 0 and "weight" in weight_df.columns
        else np.nan
    )
    max_feature_weight = (
        float(weight_df["weight"].max())
        if len(weight_df) > 0 and "weight" in weight_df.columns
        else np.nan
    )
    n_weighted_features = (
        int((weight_df["weight"] > FEATURE_WEIGHT_DEFAULT).sum())
        if len(weight_df) > 0 and "weight" in weight_df.columns
        else 0
    )

    X_train_s = X_train_t[selected].copy()
    X_test_s = X_test_t[selected].copy()

    X_train_w = apply_feature_weights_to_matrix(X_train_s, weight_df)
    X_test_w = apply_feature_weights_to_matrix(X_test_s, weight_df)

    pred_train = fitted_model.predict(X_train_w)
    pred_test = fitted_model.predict(X_test_w)

    pred_df = pd.DataFrame({
        "Patient": X_test.index,
        "y_true": y_test.values,
        "y_pred_test": pred_test,
    })

    train_r2 = r2_score(y_train, pred_train)
    train_mae = mean_absolute_error(y_train, pred_train)

    test_r2 = r2_score(y_test, pred_test)
    test_mae = mean_absolute_error(y_test, pred_test)

    summary_row = {
        "fusion_type": "EF",
        "train_experiment_tag": TRAIN_EXPERIMENT_TAG,
        "test_experiment_tag": TEST_EXPERIMENT_TAG,
        "model_type": model_type,
        "clinical_mode": clinical_mode,
        "tissue": tissue,
        "combo": combo,
        "tp": tp,
        "use_delta_view": USE_DELTA_VIEW,
        "tuning_group": get_ef_tuning_group(tissue, tp),
        "cv_search_mode": CV_SEARCH_MODE,

        "use_feature_weighting": USE_FEATURE_WEIGHTING,
        "feature_weight_modes": ",".join(FEATURE_WEIGHT_MODES),
        "feature_weight_mode": feature_weight_mode,
        "feature_selection_mode": feature_selection_mode,
        "feature_weight_min": FEATURE_WEIGHT_MIN,
        "feature_weight_max": FEATURE_WEIGHT_MAX,
        "mean_feature_weight": mean_feature_weight,
        "max_feature_weight": max_feature_weight,
        "n_weighted_features": n_weighted_features,

        "n_train_raw": len(X_train_raw),
        "n_train_after_row_filter": len(X_train),
        "n_test": len(X_test),
        "n_train_rows_dropped_all_missing_omics": row_info.get("n_removed", row_info.get("n_dropped_all_missing_omics_rows", np.nan)),
        "sparse_chosen_thresh": sparse_info.get("chosen_thresh", np.nan),
        "n_omics_after_sparse": sparse_info.get("n_omics_after", np.nan),
        "n_features_before_selection": X_train_t.shape[1],
        "n_selected_features": len(selected),

        "inner_best_selection_score": float(search_result.get("best_score", np.nan)),
        "inner_best_mean_r2": float(search_result.get("best_mean_inner_r2", np.nan)),
        "inner_best_std_r2": float(search_result.get("best_std_inner_r2", np.nan)),
        "inner_best_median_r2": float(search_result.get("best_median_inner_r2", np.nan)),
        "inner_best_min_r2": float(search_result.get("best_min_inner_r2", np.nan)),
        "inner_cv_type": search_result.get("inner_cv_type", ""),

        "train_r2": train_r2,
        "train_mae": train_mae,
        "test_r2": test_r2,
        "test_mae": test_mae,
        "best_params": json.dumps(search_result["best_params"], default=str),
    }

    return {
        "mode": "heldout_test",
        "model_type": model_type,
        "clinical_mode": clinical_mode,
        "summary_row": summary_row,
        "selected_feature_df": pd.DataFrame({
            "feature": selected,
            "rank": np.arange(1, len(selected) + 1),
        }),
        "feature_score_df": search_result.get("feature_score_df", pd.DataFrame()),
        "feature_weight_df": search_result.get("feature_weight_df", pd.DataFrame()),
        "search_df": search_result.get("search_df", pd.DataFrame()),
        "pred_df": pred_df,
    }



In [23]:
# ==========================================
# Run EF held-out TEST only for training-CV-selected configs
# ==========================================
y_all = load_target(TARGET_PATH)

if not TRAIN_SUMMARY_PATH.exists():
    raise FileNotFoundError(
        f"Training summary not found: {TRAIN_SUMMARY_PATH}\n"
        f"Check TRAIN_RESULT_DIR and training output filename."
    )

train_summary = pd.read_csv(TRAIN_SUMMARY_PATH)

ok = train_summary.copy()
if "status" in ok.columns:
    ok = ok[ok["status"].astype(str).str.lower().eq("ok")].copy()

required_cols = ["tissue", "combo", "tp", "model_type", "oof_r2"]
missing_cols = [c for c in required_cols if c not in ok.columns]
if missing_cols:
    raise ValueError(
        f"Training summary missing required columns: {missing_cols}\n"
        f"Available columns: {ok.columns.tolist()}"
    )

if "clinical_mode" not in ok.columns:
    ok["clinical_mode"] = "light"

# ==========================================
# Training-CV OOF-only config selection
# ==========================================
# Do not use test_r2 for config selection.
# Do not use train_r2/final_full_train_r2 for config selection.
# Select candidates only from v13 training-CV OOF metrics.
selection_score_col = TEST_SELECTION_PRIMARY_METRIC

if selection_score_col not in ok.columns:
    raise ValueError(
        f"Selection metric {selection_score_col!r} not found in training summary. "
        f"Available columns: {ok.columns.tolist()}"
    )

sort_cols = [selection_score_col]
ascending = [False]

# Tie-breakers only; primary selection is OOF R2.
for tie_col, asc in [
    ("mean_valid_r2", False),
    ("median_valid_r2", False),
    ("min_valid_r2", False),
    ("oof_mae", True),
]:
    if tie_col in ok.columns:
        sort_cols.append(tie_col)
        ascending.append(asc)

if TEST_CONFIG_SELECTION_MODE == "best_per_tissue_tp":
    selected_configs = (
        ok.sort_values(
            by=["tissue", "tp"] + sort_cols,
            ascending=[True, True] + ascending,
            na_position="last",
        )
        .groupby(["tissue", "tp"], as_index=False)
        .head(1)
        .reset_index(drop=True)
    )
elif TEST_CONFIG_SELECTION_MODE == "top_n_overall":
    selected_configs = (
        ok.sort_values(
            by=sort_cols,
            ascending=ascending,
            na_position="last",
        )
        .head(N_TEST_CONFIGS_OVERALL)
        .reset_index(drop=True)
    )
else:
    raise ValueError(f"Unknown TEST_CONFIG_SELECTION_MODE: {TEST_CONFIG_SELECTION_MODE}")

selected_configs["selection_score_column"] = selection_score_col
selected_configs["selection_basis"] = (
    f"{TEST_CONFIG_SELECTION_MODE}_selected_by_v13_training_oof_r2_only_not_test_r2"
)

display_cols = [
    c for c in [
        "tissue", "combo", "tp", "model_type", "clinical_mode",
        "oof_r2",
        "mean_valid_r2", "median_valid_r2", "mean_train_r2", "oof_mae",
        "selected_feature_selection_modes", "selected_feature_weight_modes",
        "selection_score_column", "selection_basis",
    ]
    if c in selected_configs.columns
]
display(selected_configs[display_cols])

summary_rows = []
all_pred_rows = []
all_selected_feature_rows = []
all_feature_score_rows = []
all_feature_weight_rows = []
all_search_rows = []

for _, row in selected_configs.iterrows():
    tissue = row["tissue"]
    combo = row["combo"]
    tp = int(row["tp"])
    model_type = row["model_type"]
    clinical_mode = row.get("clinical_mode", "light")

    print(
        f"\n[EF-HELDOUT-TEST-CVSELECTED] "
        f"tissue={tissue}, combo={combo}, tp={tp}, "
        f"model={model_type}, clinical={clinical_mode}"
    )

    ef_tag = f"EF_CVSELECTED_{model_type}_{clinical_mode}_{tissue}_{combo}_tp{tp}"

    try:
        ef_result = run_early_fusion_test_on_predefined_split(
            tissue=tissue,
            combo=combo,
            tp=tp,
            y_all=y_all,
            train_ids=TRAIN_IDS,
            test_ids=TEST_IDS,
            model_type=model_type,
            clinical_mode=clinical_mode,
        )

        summary_row = ef_result["summary_row"].copy()
        summary_row["status"] = "ok"
        summary_row["tag"] = ef_tag

        # Training-CV metrics copied from the selected row for traceability.
        summary_row["cv_oof_r2"] = row.get("oof_r2", np.nan)
        summary_row["cv_mean_valid_r2"] = row.get("mean_valid_r2", np.nan)
        summary_row["cv_median_valid_r2"] = row.get("median_valid_r2", np.nan)
        summary_row["cv_oof_mae"] = row.get("oof_mae", np.nan)
        summary_row["cv_stability_score"] = row.get("cv_stability_score", row.get("stability_score", np.nan))
        summary_row["cv_selection_score_column"] = row.get("selection_score_column", "")
        summary_row["cv_mean_train_r2"] = row.get("mean_train_r2", np.nan)
        summary_row["cv_generalization_gap_abs"] = row.get("generalization_gap_abs", np.nan)
        summary_row["cv_valid_r2_iqr"] = row.get("valid_r2_iqr", np.nan)
        summary_row["cv_n_low_sst_folds"] = row.get("n_low_sst_folds", np.nan)
        summary_row["cv_selected_feature_selection_modes"] = row.get("selected_feature_selection_modes", "")
        summary_row["cv_selected_feature_weight_modes"] = row.get("selected_feature_weight_modes", "")
        summary_row["selection_basis"] = row["selection_basis"]
        summary_row["selection_warning"] = (
            "Final candidate selected by training CV only. "
            "Do not select or rank final model by test_r2."
        )

        summary_rows.append(summary_row)

        pred_df = ef_result["pred_df"].copy()
        pred_df["tag"] = ef_tag
        pred_df["train_experiment_tag"] = TRAIN_EXPERIMENT_TAG
        pred_df["test_experiment_tag"] = TEST_EXPERIMENT_TAG
        pred_df["tissue"] = tissue
        pred_df["combo"] = combo
        pred_df["tp"] = tp
        pred_df["model_type"] = model_type
        pred_df["clinical_mode"] = clinical_mode
        pred_df["selection_basis"] = row["selection_basis"]
        all_pred_rows.append(pred_df)

        selected_feature_df = ef_result["selected_feature_df"].copy()
        if len(selected_feature_df) > 0:
            selected_feature_df["tag"] = ef_tag
            selected_feature_df["train_experiment_tag"] = TRAIN_EXPERIMENT_TAG
            selected_feature_df["test_experiment_tag"] = TEST_EXPERIMENT_TAG
            selected_feature_df["tissue"] = tissue
            selected_feature_df["combo"] = combo
            selected_feature_df["tp"] = tp
            selected_feature_df["model_type"] = model_type
            selected_feature_df["clinical_mode"] = clinical_mode
            selected_feature_df["selection_basis"] = row["selection_basis"]
            all_selected_feature_rows.append(selected_feature_df)

        feature_score_df = ef_result["feature_score_df"].copy()
        if len(feature_score_df) > 0:
            feature_score_df["tag"] = ef_tag
            feature_score_df["train_experiment_tag"] = TRAIN_EXPERIMENT_TAG
            feature_score_df["test_experiment_tag"] = TEST_EXPERIMENT_TAG
            feature_score_df["tissue"] = tissue
            feature_score_df["combo"] = combo
            feature_score_df["tp"] = tp
            feature_score_df["model_type"] = model_type
            feature_score_df["clinical_mode"] = clinical_mode
            feature_score_df["selection_basis"] = row["selection_basis"]
            all_feature_score_rows.append(feature_score_df)

        feature_weight_df = ef_result["feature_weight_df"].copy()
        if len(feature_weight_df) > 0:
            feature_weight_df["tag"] = ef_tag
            feature_weight_df["train_experiment_tag"] = TRAIN_EXPERIMENT_TAG
            feature_weight_df["test_experiment_tag"] = TEST_EXPERIMENT_TAG
            feature_weight_df["tissue"] = tissue
            feature_weight_df["combo"] = combo
            feature_weight_df["tp"] = tp
            feature_weight_df["model_type"] = model_type
            feature_weight_df["clinical_mode"] = clinical_mode
            feature_weight_df["selection_basis"] = row["selection_basis"]
            all_feature_weight_rows.append(feature_weight_df)

        search_df = ef_result["search_df"].copy()
        if len(search_df) > 0:
            search_df["tag"] = ef_tag
            search_df["train_experiment_tag"] = TRAIN_EXPERIMENT_TAG
            search_df["test_experiment_tag"] = TEST_EXPERIMENT_TAG
            search_df["tissue"] = tissue
            search_df["combo"] = combo
            search_df["tp"] = tp
            search_df["model_type"] = model_type
            search_df["clinical_mode"] = clinical_mode
            search_df["selection_basis"] = row["selection_basis"]
            all_search_rows.append(search_df)

    except Exception as e:
        summary_rows.append({
            "status": "failed",
            "fusion_type": "EF",
            "train_experiment_tag": TRAIN_EXPERIMENT_TAG,
            "test_experiment_tag": TEST_EXPERIMENT_TAG,
            "use_feature_weighting": USE_FEATURE_WEIGHTING,
            "feature_weight_modes": ",".join(FEATURE_WEIGHT_MODES),
            "tissue": tissue,
            "combo": combo,
            "tp": tp,
            "model_type": model_type,
            "clinical_mode": clinical_mode,
            "tag": ef_tag,
            "cv_oof_r2": row.get("oof_r2", np.nan),
            "cv_mean_valid_r2": row.get("mean_valid_r2", np.nan),
            "cv_median_valid_r2": row.get("median_valid_r2", np.nan),
            "cv_oof_mae": row.get("oof_mae", np.nan),
            "cv_stability_score": row.get("cv_stability_score", row.get("stability_score", np.nan)),
            "cv_selection_score_column": row.get("selection_score_column", ""),
            "cv_mean_train_r2": row.get("mean_train_r2", np.nan),
            "cv_generalization_gap_abs": row.get("generalization_gap_abs", np.nan),
            "cv_valid_r2_iqr": row.get("valid_r2_iqr", np.nan),
            "cv_n_low_sst_folds": row.get("n_low_sst_folds", np.nan),
            "selection_basis": row["selection_basis"],
            "test_r2": np.nan,
            "test_mae": np.nan,
            "train_r2": np.nan,
            "train_mae": np.nan,
            "error_message": traceback.format_exc(),
        })

        print(
            f"[EF-HELDOUT-TEST-FAILED] "
            f"tissue={tissue}, combo={combo}, tp={tp}, "
            f"model={model_type}, clinical={clinical_mode} :: {e}"
        )

summary_df = pd.DataFrame(summary_rows).sort_values(
    by=["status", "tissue", "tp", "model_type", "clinical_mode"],
    ascending=[True, True, True, True, True],
    na_position="last",
).reset_index(drop=True)

summary_path = SAVE_DIR / f"multi_omics_ef_holdout_test_summary_{TEST_EXPERIMENT_TAG}.csv"
selected_path = SAVE_DIR / f"multi_omics_ef_cv_selected_configs_used_for_test_{TEST_EXPERIMENT_TAG}.csv"

summary_df.to_csv(summary_path, index=False)
selected_configs.to_csv(selected_path, index=False)


# Paper-style held-out test table: one selected/trained config per tissue + TP.
# This table is for reporting held-out performance only; selection was already fixed by v13 training OOF.
paper_style_cols = [
    c for c in [
        "fusion_type", "tissue", "tp", "combo", "model_type", "clinical_mode", "tuning_group",
        "cv_oof_r2", "cv_mean_valid_r2", "cv_median_valid_r2", "cv_oof_mae",
        "train_r2", "train_mae", "test_r2", "test_mae",
        "n_train_after_row_filter", "n_test", "n_selected_features",
        "feature_selection_mode", "feature_weight_mode", "best_params",
        "selection_basis",
    ]
    if c in summary_df.columns
]
paper_style_test_table = (
    summary_df[summary_df["status"].astype(str).str.lower().eq("ok")]
    .copy()
    .sort_values(["tissue", "tp"])
    .reset_index(drop=True)
)

paper_style_test_path = SAVE_DIR / f"paper_style_best_by_tissue_tp_test_{TEST_EXPERIMENT_TAG}.csv"
paper_style_test_table[paper_style_cols].to_csv(paper_style_test_path, index=False)

# Extra grouped view by tissue + TP + model + clinical mode for checking.
grouped_test_path = SAVE_DIR / f"grouped_by_tissue_tp_model_clinical_test_{TEST_EXPERIMENT_TAG}.csv"
if len(paper_style_test_table) > 0:
    grouped_test = (
        paper_style_test_table
        .groupby(["tissue", "tp", "model_type", "clinical_mode"], dropna=False)
        .agg(
            n=("test_r2", "size"),
            cv_oof_r2=("cv_oof_r2", "mean"),
            test_r2=("test_r2", "mean"),
            test_mae=("test_mae", "mean"),
            train_r2=("train_r2", "mean"),
            train_mae=("train_mae", "mean"),
        )
        .reset_index()
        .sort_values(["tissue", "tp", "test_r2"], ascending=[True, True, False])
    )
    grouped_test.to_csv(grouped_test_path, index=False)


if all_pred_rows:
    pd.concat(all_pred_rows, ignore_index=True).to_csv(
        SAVE_DIR / f"multi_omics_ef_holdout_test_predictions_{TEST_EXPERIMENT_TAG}.csv",
        index=False,
    )

if all_selected_feature_rows:
    pd.concat(all_selected_feature_rows, ignore_index=True).to_csv(
        SAVE_DIR / f"multi_omics_ef_holdout_test_selected_features_{TEST_EXPERIMENT_TAG}.csv",
        index=False,
    )

if all_feature_score_rows:
    pd.concat(all_feature_score_rows, ignore_index=True).to_csv(
        SAVE_DIR / f"multi_omics_ef_holdout_test_feature_scores_{TEST_EXPERIMENT_TAG}.csv",
        index=False,
    )

if all_feature_weight_rows:
    pd.concat(all_feature_weight_rows, ignore_index=True).to_csv(
        SAVE_DIR / f"multi_omics_ef_holdout_test_feature_weights_{TEST_EXPERIMENT_TAG}.csv",
        index=False,
    )

if all_search_rows:
    pd.concat(all_search_rows, ignore_index=True).to_csv(
        SAVE_DIR / f"multi_omics_ef_holdout_test_search_results_{TEST_EXPERIMENT_TAG}.csv",
        index=False,
    )

display(summary_df)
print("Saved EF held-out test summary to:", summary_path)
print("Saved selected CV configs to:", selected_path)
print("Saved paper-style held-out test table to:", paper_style_test_path)
print("Saved grouped held-out test check to:", grouped_test_path)
print("Saved all EF held-out test outputs to:", SAVE_DIR)



,tissue,combo,tp,model_type,clinical_mode,oof_r2,mean_valid_r2,median_valid_r2,mean_train_r2,oof_mae,selected_feature_selection_modes,selected_feature_weight_modes,selection_score_column,selection_basis
0,csf,ABC,24,elasticnet,omics_only,0.063760,0.027309,0.077321,0.700874,16.365680,f_regression_topk,none,oof_r2,best_per_tissue_tp_selected_by_v13_training_oo...
1,csf,ABC,48,ridge,omics_only,0.051320,-0.012800,0.172116,0.616196,15.018140,f_regression_topk,none,oof_r2,best_per_tissue_tp_selected_by_v13_training_oo...
2,csf,ABC,72,elasticnet,omics_only,0.214499,0.182474,0.205964,0.716135,14.560225,f_regression_topk,"none,soft",oof_r2,best_per_tissue_tp_selected_by_v13_training_oo...
3,csf,ABC,96,gbr,omics_only,0.217536,0.225393,0.115847,0.823585,12.929825,f_regression_topk,none,oof_r2,best_per_tissue_tp_selected_by_v13_training_oo...
4,csf,ABC,120,ridge,light,0.040638,-0.012915,-0.024963,0.917533,14.993438,f_regression_topk,"none,soft",oof_r2,best_per_tissue_tp_selected_by_v13_training_oo...
5,ser,ABC,24,ridge,omics_only,-0.081395,-0.101271,-0.095151,0.584206,17.768764,f_regression_topk,none,oof_r2,best_per_tissue_tp_selected_by_v13_training_oo...
6,ser,ABC,48,ridge,omics_only,-0.002291,0.008528,0.004533,0.580475,16.797144,f_regression_topk,"none,soft",oof_r2,best_per_tissue_tp_selected_by_v13_training_oo...
7,ser,ABC,72,ridge,light,0.003662,0.040275,0.109404,0.501259,16.914001,f_regression_topk,"none,soft",oof_r2,best_per_tissue_tp_selected_by_v13_training_oo...
8,ser,ABC,96,gbr,light,0.163245,0.206926,0.359867,0.730235,15.352564,f_regression_topk,none,oof_r2,best_per_tissue_tp_selected_by_v13_training_oo...
9,ser,ABC,120,ridge,light,0.089067,0.114322,0.103931,0.636490,15.483793,f_regression_topk,"none,soft",oof_r2,best_per_tissue_tp_selected_by_v13_training_oo...



[EF-HELDOUT-TEST-CVSELECTED] tissue=csf, combo=ABC, tp=24, model=elasticnet, clinical=omics_only
[DEBUG][BLOCKWISE] counts = {'PROT': 250, 'MET': 459, 'RNA': 258, 'CLIN': 0}
[DEBUG][BLOCKWISE] counts = {'PROT': 250, 'MET': 459, 'RNA': 258, 'CLIN': 0}
[DEBUG][BLOCKWISE] counts = {'PROT': 250, 'MET': 459, 'RNA': 258, 'CLIN': 0}
[DEBUG][BLOCKWISE] counts = {'PROT': 250, 'MET': 459, 'RNA': 258, 'CLIN': 0}
[DEBUG][BLOCKWISE] counts = {'PROT': 250, 'MET': 459, 'RNA': 258, 'CLIN': 0}
[DEBUG][BLOCKWISE] counts = {'PROT': 250, 'MET': 459, 'RNA': 258, 'CLIN': 0}
[DEBUG][BLOCKWISE] counts = {'PROT': 250, 'MET': 459, 'RNA': 258, 'CLIN': 0}
[DEBUG][BLOCKWISE] counts = {'PROT': 250, 'MET': 459, 'RNA': 258, 'CLIN': 0}
[DEBUG][BLOCKWISE] counts = {'PROT': 250, 'MET': 459, 'RNA': 258, 'CLIN': 0}
[DEBUG][BLOCKWISE] counts = {'PROT': 250, 'MET': 459, 'RNA': 258, 'CLIN': 0}
[DEBUG][BLOCKWISE] counts = {'PROT': 250, 'MET': 459, 'RNA': 258, 'CLIN': 0}
[DEBUG][BLOCKWISE] counts = {'PROT': 250, 'MET': 459, '

,fusion_type,train_experiment_tag,test_experiment_tag,model_type,clinical_mode,tissue,combo,tp,use_delta_view,tuning_group,...,cv_stability_score,cv_selection_score_column,cv_mean_train_r2,cv_generalization_gap_abs,cv_valid_r2_iqr,cv_n_low_sst_folds,cv_selected_feature_selection_modes,cv_selected_feature_weight_modes,selection_basis,selection_warning
0,EF,ef_paper_style_v13_tp_model_refined_tuning,ef_final_test_paper_style_v13_tp_model_refined...,elasticnet,omics_only,csf,ABC,24,False,middle,...,NaN,oof_r2,0.700874,NaN,NaN,0,f_regression_topk,none,best_per_tissue_tp_selected_by_v13_training_oo...,Final candidate selected by training CV only. ...
1,EF,ef_paper_style_v13_tp_model_refined_tuning,ef_final_test_paper_style_v13_tp_model_refined...,ridge,omics_only,csf,ABC,48,False,middle,...,NaN,oof_r2,0.616196,NaN,NaN,0,f_regression_topk,none,best_per_tissue_tp_selected_by_v13_training_oo...,Final candidate selected by training CV only. ...
2,EF,ef_paper_style_v13_tp_model_refined_tuning,ef_final_test_paper_style_v13_tp_model_refined...,elasticnet,omics_only,csf,ABC,72,False,strong,...,NaN,oof_r2,0.716135,NaN,NaN,0,f_regression_topk,"none,soft",best_per_tissue_tp_selected_by_v13_training_oo...,Final candidate selected by training CV only. ...
3,EF,ef_paper_style_v13_tp_model_refined_tuning,ef_final_test_paper_style_v13_tp_model_refined...,gbr,omics_only,csf,ABC,96,False,strong,...,NaN,oof_r2,0.823585,NaN,NaN,0,f_regression_topk,none,best_per_tissue_tp_selected_by_v13_training_oo...,Final candidate selected by training CV only. ...
4,EF,ef_paper_style_v13_tp_model_refined_tuning,ef_final_test_paper_style_v13_tp_model_refined...,ridge,light,csf,ABC,120,False,middle,...,NaN,oof_r2,0.917533,NaN,NaN,0,f_regression_topk,"none,soft",best_per_tissue_tp_selected_by_v13_training_oo...,Final candidate selected by training CV only. ...
5,EF,ef_paper_style_v13_tp_model_refined_tuning,ef_final_test_paper_style_v13_tp_model_refined...,ridge,omics_only,ser,ABC,24,False,weak,...,NaN,oof_r2,0.584206,NaN,NaN,0,f_regression_topk,none,best_per_tissue_tp_selected_by_v13_training_oo...,Final candidate selected by training CV only. ...
6,EF,ef_paper_style_v13_tp_model_refined_tuning,ef_final_test_paper_style_v13_tp_model_refined...,ridge,omics_only,ser,ABC,48,False,weak,...,NaN,oof_r2,0.580475,NaN,NaN,0,f_regression_topk,"none,soft",best_per_tissue_tp_selected_by_v13_training_oo...,Final candidate selected by training CV only. ...
7,EF,ef_paper_style_v13_tp_model_refined_tuning,ef_final_test_paper_style_v13_tp_model_refined...,ridge,light,ser,ABC,72,False,weak,...,NaN,oof_r2,0.501259,NaN,NaN,0,f_regression_topk,"none,soft",best_per_tissue_tp_selected_by_v13_training_oo...,Final candidate selected by training CV only. ...
8,EF,ef_paper_style_v13_tp_model_refined_tuning,ef_final_test_paper_style_v13_tp_model_refined...,gbr,light,ser,ABC,96,False,strong,...,NaN,oof_r2,0.730235,NaN,NaN,0,f_regression_topk,none,best_per_tissue_tp_selected_by_v13_training_oo...,Final candidate selected by training CV only. ...
9,EF,ef_paper_style_v13_tp_model_refined_tuning,ef_final_test_paper_style_v13_tp_model_refined...,ridge,light,ser,ABC,120,False,middle,...,NaN,oof_r2,0.636490,NaN,NaN,0,f_regression_topk,"none,soft",best_per_tissue_tp_selected_by_v13_training_oo...,Final candidate selected by training CV only. ...


Saved EF held-out test summary to: ../../DifferentCom_data_rebuild/testResult/results_multi_omics_ef_test_ef_final_test_paper_style_v13_tp_model_refined_tuning/multi_omics_ef_holdout_test_summary_ef_final_test_paper_style_v13_tp_model_refined_tuning.csv
Saved selected CV configs to: ../../DifferentCom_data_rebuild/testResult/results_multi_omics_ef_test_ef_final_test_paper_style_v13_tp_model_refined_tuning/multi_omics_ef_cv_selected_configs_used_for_test_ef_final_test_paper_style_v13_tp_model_refined_tuning.csv
Saved paper-style held-out test table to: ../../DifferentCom_data_rebuild/testResult/results_multi_omics_ef_test_ef_final_test_paper_style_v13_tp_model_refined_tuning/paper_style_best_by_tissue_tp_test_ef_final_test_paper_style_v13_tp_model_refined_tuning.csv
Saved grouped held-out test check to: ../../DifferentCom_data_rebuild/testResult/results_multi_omics_ef_test_ef_final_test_paper_style_v13_tp_model_refined_tuning/grouped_by_tissue_tp_model_clinical_test_ef_final_test_paper_